# Recalibración de CO₂ v2: LAB016–021, química fija y muestras corregidas de nitrógeno

**Actualización: LAB019–021 leen directamente `YAN_muestras_corregidas.csv`. La corrección se realizó fuera del notebook, en ese archivo de muestras. El modelo y sus ecuaciones no contienen ningún offset ni vuelven a descontar 30 mg/L.**

Se ajustan exclusivamente registros gaseosos de LAB016–021. Los datos de CO₂ de LAB001–015 **no entran en ninguna función objetivo**. La calibración histórica se conserva como referencia e inicialización, sin reemplazar sus archivos ni los parámetros de producción.

Se comparan `theta_full` y `theta_defaults` (los parámetros químicos antiguos). Las mediciones de N y el N inicial de LAB019–021 se toman del CSV corregido, en el que la preparación externa ya dejó en cero las muestras negativas tras el descuento. Se leen como datos de entrada; no se recalculan desde PAN/AMMONIA ni se corrigen dentro del modelo. LAB016–018 usan ahora la concentración viable inicial de 13×10⁶ células/mL indicada por el usuario (X₀=0.39 g/L). Se conservan las nutriciones y ecuaciones químicas/gaseosas. LAB019–021 usan los archivos completos de CO₂ y temperatura, recortados en la última muestra química de cada LAB.

LAB019–021: biomasa viable ponderada del inóculo. LAB016–018: se conservan las hipótesis aceptadas por el usuario: $X_0=0.39$ g/L, $X_{d,0}=0$, G₀+F₀=172.5 g/L a partir de la densidad inicial común 1069 g/L reportada por el usuario (G:F proporcional al proxy anterior), Gly inicial proxy del Excel, $E_0=0$ y dosis de N de 0.14/0.28/0.14 g/L. LAB017 tiene dos eventos de 0.14 g/L. No hay series directas Y15 de N/G/F/E para validar la química de LAB016–018.

La simulación abarca desde inoculación hasta el final, incluidos todos los pulsos. La activación gaseosa depende del consumo **predicho** de azúcar S=G+F, con transición progresiva entre **2 y 10 g/L consumidos**; no se desplaza la química ni se impone una condición inicial tomada en el futuro. No es todavía un EKF.

**Lectura:** primero comprobar entradas y química; después comparar ganancia sola (Z1) con liberación + solubilidad + ganancia (Z2); finalmente revisar transferencia entre corridas. Un ajuste con los seis LAB es calibración, no validación externa.

Las dos figuras químicas se muestran juntas, primero LAB019–021 y después LAB016–018, incluyendo biomasa viable medida. Los recuentos Oculyze de LAB016–018 son posteriores a t=0 y se superponen como comparación; no sustituyen el X₀=0.39 g/L indicado ahora por el usuario.


**Versión v2:** copia independiente del notebook anterior, que se conserva como respaldo junto con sus resultados. Esta versión escribe exclusivamente en `results/co2_lab016_021_recalibration_v2/`. La química y las condiciones iniciales son las mismas; se cambia la activación de la fuente gaseosa y se reajustan Z1/Z2 bajo esa nueva regla. Se conserva la simulación desde inoculación hasta el final, con todos los pulsos.


In [ ]:
from pathlib import Path
from dataclasses import replace
import hashlib
import inspect
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

%matplotlib inline
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 110)

ROOT = Path.cwd().resolve()
while ROOT.name != 'pyomo-doe' and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if ROOT.name != 'pyomo-doe':
    raise RuntimeError(f'Project root pyomo-doe not found from {Path.cwd()}')
FM = ROOT / 'fermentation_model'
LAB_DIR = FM / 'laboratory_2026'
PILOT_DIR = FM / 'pilot_2025'
DATA_DIR = FM / 'data' / 'mem2026' / 'LAB016-018'
for path in (FM, LAB_DIR, PILOT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from shared import run_new_must_glycerol_estimability_doe as base
from laboratory_2026 import run_co2_matrix_cross_validation_2026 as co2_cross

THETA_PATH = LAB_DIR / 'results' / 'estimability_historical_natural' / 'theta.csv'
HISTORICAL_BATCH_SUMMARY_PATH = LAB_DIR / 'results' / 'estimability_historical_natural' / 'batch_summary.csv'
CO2_PARAMETER_PATH = LAB_DIR / 'results' / 'co2_matrix_cross_validation_2026' / 'fit_parameters.csv'
WORKBOOK_PATH = DATA_DIR / 'LAB016-018-offline-measurements.xlsx'
PRIOR_MASK_NOTEBOOK = LAB_DIR / 'notebooks' / 'lab016_018_co2_temperature_visualization.ipynb'

LABS = {
    'LAB016': {'fermenter': 'F1', 'co2_raw': 'CO2_F1_LAB016.csv', 'co2_filt': 'CO2_FILT_F1_LAB016.csv', 'temp': 'Temp_F1_LAB016.csv'},
    'LAB017': {'fermenter': 'F2', 'co2_raw': 'CO2_F2_LAB017.csv', 'co2_filt': 'CO2_FILT_F2_LAB017.csv', 'temp': 'Temp_F2_LAB017.csv'},
    # The extra dot in the F3 temperature filename is literal on disk.
    'LAB018': {'fermenter': 'F3', 'co2_raw': 'CO2_F3_LAB018.csv', 'co2_filt': 'CO2_FILT_F3_LAB018.csv', 'temp': 'Temp_F3._LAB018.csv'},
}

def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

protected_paths = [THETA_PATH, CO2_PARAMETER_PATH, WORKBOOK_PATH]
for spec in LABS.values():
    protected_paths.extend(DATA_DIR / spec[key] for key in ('co2_raw', 'co2_filt', 'temp'))
HASHES_BEFORE = {str(path.relative_to(ROOT)): sha256(path) for path in protected_paths}
print('Python:', sys.executable)
print('Project root:', ROOT)
print('Protected input hashes recorded:', len(HASHES_BEFORE))

import ast, json, subprocess
from datetime import datetime
from scipy.optimize import least_squares, minimize_scalar
from IPython.display import Image
assert Path(sys.prefix).name.lower() == 'fermentation', sys.prefix
V1_OUT=LAB_DIR/'results/co2_lab016_021_recalibration'
V1_SOURCE_PATH=LAB_DIR/'notebooks/diagnostics/co2_lab016_021_recalibration.source.ipynb'
V1_EXECUTED_PATH=V1_SOURCE_PATH.with_name(V1_SOURCE_PATH.name.replace('.source.','.executed.'))
V1_FILES=[V1_SOURCE_PATH,V1_EXECUTED_PATH]+[p for p in V1_OUT.rglob('*') if p.is_file()]
V1_HASHES_BEFORE={str(p.relative_to(ROOT)):sha256(p) for p in V1_FILES}
OUT=LAB_DIR/'results/co2_lab016_021_recalibration_v2'
OUT.mkdir(exist_ok=True)
PRIOR=LAB_DIR/'results/nitrogen_consumption_diagnosis'
sys.path.insert(0,str(LAB_DIR/'notebooks/EKF'))
import _post_activity_prediction as gas_model
from shared import run_secondary_joint_campaign_doe as joint
CORRECTED_YAN_PATH=FM/'data/mem2026/LAB019-021/YAN_muestras_corregidas.csv'
LOD=.05
FACTOR=44.0095*60*.74/(1000*24.16*2.)
assert np.isclose(joint.CO2_G_PER_G_ETHANOL,44.01/46.07)
COHORT1=('LAB016','LAB017','LAB018')
COHORT2=('LAB019','LAB020','LAB021')
ALL_LABS=COHORT1+COHORT2
FIGURES=[]
def show(fig,name):
    if fig._suptitle is not None:
        title=fig._suptitle.get_text()
        fig._suptitle.remove();fig._suptitle=None
        fig.text(.5,.99,title,ha='center',va='top',fontsize=14,transform=fig.transFigure)
    for legend in fig.legends:
        legend.set_bbox_to_anchor((.5,.955),transform=fig.transFigure)
    fig.tight_layout(rect=(0,0,1,.90))
    fig.set_layout_engine('none')
    fig.savefig(OUT/(name+'.png'),dpi=140,bbox_inches='tight')
    FIGURES.append(name+'.png')
    display(fig)
    plt.close(fig)
GIT_HEAD=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip()
GIT_REMOTE=subprocess.check_output(['git','remote','get-url','origin'],cwd=ROOT,text=True).strip()
display(Markdown(f'Repositorio actual: `{GIT_REMOTE}`; revisión de partida: `{GIT_HEAD}`.'))
RECENT_DATA_DIR=FM/'data/mem2026/LAB019-021'
RECENT_CO2_PATH=RECENT_DATA_DIR/'backup_global_co2_completo.csv'
RECENT_TEMP_PATH=RECENT_DATA_DIR/'backup_global_temperatura_completo.csv'
RECENT_T0=pd.Timestamp('2026-09-21 17:00:00')
RECENT_FERMENTERS=dict(zip(COHORT2,['F1','F2','F3']))

SUGAR_ACTIVATION_START_G_L=2.
SUGAR_ACTIVATION_FULL_G_L=10.


## 1. Modelo y parámetros que se ajustan

Las ecuaciones utilizadas corresponden a la capa simplificada de `_post_activity_prediction.finish`, empleada en el diagnóstico de nitrógeno/CO₂. Se conserva su mezcla de trayectorias con y sin nutrición; no se confunde con el modelo original más amplio que incluye otros términos respiratorios.

$$q_n=Y_{CO_2/E}(\beta_G+\beta_F)X,\quad Y_{CO_2/E}=44.01/46.07\simeq0.9553.$$
$$S=G+F,\qquad \Delta S=S_0-S.$$
$$z=\operatorname{clip}\!\left(\frac{\Delta S-2}{10-2},0,1\right),\qquad a=z^2(3-2z).$$

La fuente gaseosa se activa progresivamente usando únicamente el **consumo predicho desde inoculación**: a=0 hasta consumir 2 g/L de S, 0<a<1 entre 2 y 10 g/L, y a=1 desde 10 g/L. En el punto medio, 6 g/L consumidos, a=0.5. El umbral se refiere a azúcar **consumido**, no al azúcar residual. No se impone una condición de etanol para habilitar la fuente; la química y sus términos cinéticos se conservan. La puerta no desplaza t=0, no reinicia estados ni usa cruces futuros para definir la activación actual.
$$r=\operatorname{clip}((t-t_{N,1})/\tau_N,0,1),\quad b=1+(b_N-1)r.$$
$$q_{bio}=M(t)\max[bq_0+r(q_n-q_0),0],\qquad q_{prod}=a\,f_{ferm}(O_2)q_{bio}.$$
Aquí $q_0$ es la producción calculada con la trayectoria sin pulsos de N. $M$ suma los transitorios de **todos** los pulsos: $1+\sum_j A(d_j/0.14)h(t-t_j)$, con $h(u)=(1-e^{-u/4})e^{-u/12}$ para $u\ge0$, normalizado por su máximo analítico. $f_{ferm}=0.08+0.92\,0.75^2/(0.75^2+O_2^2)$.

$$C_{sat}=s\,1.69 e^{-0.032(T-20)}e^{0.0016E}e^{-0.0012(G+F)},$$
$$q_{sal}=k C_d\left(0.05+0.95\frac{C_d}{C_d+C_{sat}}\right),\quad \dot C_d=q_{prod}-q_{sal},\quad \hat y=gq_{sal}.$$

Se usa el balance discreto conservativo ya existente, con $C_d(0)=0$ y oxígeno inicial calculado desde la solubilidad. El O₂ evoluciona según la captación existente, cuyos parámetros se mantienen fijos.

| Ajuste | Vector libre | Se mantiene fijo |
|---|---|---|
| Z1 | $g$ = `matrix_gain` | $k,s$, química, rendimiento, activación, O₂, pulsos |
| Z2 | $k,s,g$ = `kCO2_release_h`, `CO2sat_scale`, `matrix_gain` | química, rendimiento, activación, O₂, pulsos |

El rendimiento 0.9553 **no se calibra** y es idéntico para ambas theta. Tampoco se ajusta la conversión SCCM → g/L/h. `matrix_gain` es una ganancia efectiva; no debe interpretarse automáticamente como rendimiento estequiométrico.

**Procedencia:** el remoto actual es `crodv/fermentation-model`. El proyecto conserva antecedentes de `dowlinglab/pyomo-doe`, que proporciona infraestructura de Pyomo/DoE, no una calibración de estas fermentaciones. El historial local incorpora el modelo de fermentación en `79fb549a87` y el pipeline de ajuste de CO₂ en `579d6f7c72`. Aquí se registra además el SHA actual. La tabla antigua se lee sólo para obtener valores iniciales; no se cargan sus observaciones de CO₂.

In [ ]:
full_path=LAB_DIR/'results/estimability_historical_natural/theta_natural_full.csv'
full=pd.read_csv(full_path).set_index('parameter').theta.astype(float).to_dict()
partial=pd.read_csv(THETA_PATH).set_index('parameter').theta.astype(float).to_dict()
THETAS={'theta_full':{p:full[p] for p in base.FULL17},
        'theta_defaults':{p:{**base.DEFAULT_THETA,**partial}[p] for p in base.FULL17}}
layers=pd.read_csv(CO2_PARAMETER_PATH)
layer=layers[layers.calibration_matrix.eq('natural')]
OLD=dict(zip(layer.parameter,layer.estimate.astype(float)))
OLD.update(A=.30,tau_rise_h=4.,tau_decay_h=12.,N_ref_g_l=.14,kernel_normalization='analytic_peak')
BOUNDS={'kCO2_release_h':(.03,25.),'CO2sat_scale':(.35,2.5),'matrix_gain':(.1,20.)}
display(pd.DataFrame(THETAS))
display(pd.DataFrame([{'parameter':p,'historical_start':OLD[p],'lower':b[0],'upper':b[1]} for p,b in BOUNDS.items()]))
print('Y_CO2/E fijo =',joint.CO2_G_PER_G_ETHANOL,'; conversión SCCM =',FACTOR)
protected_paths += [full_path,Path(gas_model.__file__),Path(base.__file__),Path(joint.__file__)]
THETA_SNAPSHOT=json.loads(json.dumps(THETAS))
OLD_SNAPSHOT=OLD.copy()
display(pd.DataFrame([dict(parameter=k,value=v) for k,v in OLD.items() if k not in BOUNDS]))

## 2. LAB016–018: datos y auditoría de artefactos

Se reproducen las reglas del notebook `lab016_018_natural_must_holdout`: reloj en el fin de inoculación, detección de pulsos de bomba y máscara del transitorio del filtro (90 min de recuperación). Se conserva el criterio, no sus ajustes de parámetros ni sus curvas predichas. Las dosis posteriores se aplican en el inicio del evento registrado.

Para la corrección de cero se usa, en los seis LAB, el percentil 10 del caudal filtrado válido entre el final del artefacto inicial y 12 h; se resta y se limita el caudal corregido a cero. Es un procesamiento de datos fijado antes de optimizar, no otro parámetro libre. La predicción causal no utiliza esta ventana para definir su activación.
Para LAB019–021 se leen ahora los registros completos y se conserva la máscara inicial, la corrección de cero y la conversión del sensor. Se agregan medianas horarias sólo hasta la última muestra química; no se extiende la curva hasta el final del archivo completo.


In [ ]:
excel = pd.ExcelFile(WORKBOOK_PATH)
offline = pd.read_excel(WORKBOOK_PATH, sheet_name='MedicionesOffline')
experiments = pd.read_excel(WORKBOOK_PATH, sheet_name='Experimentos')
initial_sheet = pd.read_excel(WORKBOOK_PATH, sheet_name='CondicionesIniciales')
offline_csv = pd.read_csv(DATA_DIR / 'LAB016-018-offline-measurements.csv')

signals = {}
inventory_rows = []
for lab, spec in LABS.items():
    signals[lab] = {
        'fermenter': spec['fermenter'],
        'co2_raw': pd.read_csv(DATA_DIR / spec['co2_raw'], parse_dates=['timestamp']),
        'co2_filt': pd.read_csv(DATA_DIR / spec['co2_filt'], parse_dates=['timestamp']),
        'temp': pd.read_csv(DATA_DIR / spec['temp'], parse_dates=['timestamp']),
    }
    for key in ('co2_raw', 'co2_filt', 'temp'):
        frame = signals[lab][key]
        inventory_rows.append({
            'LAB': lab, 'fermenter': spec['fermenter'], 'kind': key, 'file': spec[key],
            'rows': len(frame), 'start': frame['timestamp'].min(), 'end': frame['timestamp'].max(),
            'columns': ', '.join(frame.columns),
        })
sensor_inventory = pd.DataFrame(inventory_rows)
observable_inventory = pd.DataFrame([
    {'available_quantity': 'filtered CO2 flow (SCCM)', 'source': 'CO2_FILT_*', 'model correspondence': 'CO2 layer output after official in-memory unit conversion', 'used in holdout metrics': True},
    {'available_quantity': 'reactor temperature (degC)', 'source': 'Temp_*: T', 'model correspondence': 'known exogenous temperature input, not a predicted state', 'used in holdout metrics': False},
    {'available_quantity': 'Brix', 'source': 'MedicionesOffline', 'model correspondence': 'none validated for G/F/G+F', 'used in holdout metrics': False},
    {'available_quantity': 'density (g/L)', 'source': 'MedicionesOffline', 'model correspondence': 'none validated for G/F/G+F', 'used in holdout metrics': False},
    {'available_quantity': 'sample temperature / residual volume', 'source': 'MedicionesOffline', 'model correspondence': 'QC/context only', 'used in holdout metrics': False},
    {'available_quantity': 'viable / total biomass', 'source': 'LAB016_OCULYZE/report.csv and LAB017/LAB018 equivalents', 'model correspondence': 'viable count x 0.03 g/L per million cells/mL; comparison only', 'used in holdout metrics': False},
    {'available_quantity': 'N/YAN, G, F, E, Gly, O2 time series', 'source': 'not present as direct chemistry assays', 'model correspondence': 'core states', 'used in holdout metrics': False},
])
print('Workbook sheets:', excel.sheet_names)
print('Offline Excel rows:', len(offline), '| duplicate CSV rows:', len(offline_csv))
display(sensor_inventory)
display(observable_inventory)
display(initial_sheet)

In [ ]:
def combine_date_time(date_value, time_value):
    date_part = pd.Timestamp(date_value).date()
    if not hasattr(time_value, 'hour'):
        time_value = pd.to_datetime(str(time_value)).time()
    return pd.Timestamp.combine(date_part, time_value)

initial_sheet['timestamp_inoculation'] = initial_sheet.apply(
    lambda row: combine_date_time(row['Fecha'], row['Hora']), axis=1
)
T0_BY_LAB = dict(zip(initial_sheet['CodigoLab'], initial_sheet['timestamp_inoculation']))

HAMPEL_WINDOW_S = 120.0
HAMPEL_K = 3.5
HAMPEL_MIN_ABS_SCCM = 0.5
IIR_TAU_S = 1200.0
IIR_ORDER = 3

def extract_binary_pulses(lab, temp):
    state = pd.to_numeric(temp['nutricion_activa'], errors='raise').astype(int)
    if not set(state.unique()).issubset({0, 1}):
        raise ValueError(f'{lab}: nutricion_activa is not binary')
    rising = state.eq(1) & ~state.shift(fill_value=0).eq(1)
    falling = state.eq(0) & state.shift(fill_value=0).eq(1)
    starts = np.flatnonzero(rising.to_numpy())
    falls = np.flatnonzero(falling.to_numpy())
    median_dt_s = float(temp['timestamp'].diff().dt.total_seconds().median())
    rows = []
    for number, start_i in enumerate(starts, start=1):
        fall_position = np.searchsorted(falls, start_i + 1)
        has_fall = fall_position < len(falls)
        start = temp['timestamp'].iloc[start_i]
        end = temp['timestamp'].iloc[falls[fall_position]] if has_fall else temp['timestamp'].iloc[-1] + pd.to_timedelta(median_dt_s, unit='s')
        rows.append({
            'LAB': lab, 'fermenter': LABS[lab]['fermenter'], 'pulse_number': number,
            'rising_edge_observed': bool(start_i > 0), 'falling_edge_observed': bool(has_fall),
            'pulse_start': start, 'pulse_end': end,
            'duration_min': (end - start).total_seconds() / 60.0,
        })
    return pd.DataFrame(rows)

pulse_table = pd.concat([extract_binary_pulses(lab, signals[lab]['temp']) for lab in LABS], ignore_index=True)
pulse_table['t0'] = pulse_table['LAB'].map(T0_BY_LAB)
pulse_table['pulse_start_h'] = (pulse_table['pulse_start'] - pulse_table['t0']).dt.total_seconds() / 3600.0
pulse_table['pulse_end_h'] = (pulse_table['pulse_end'] - pulse_table['t0']).dt.total_seconds() / 3600.0

def classify_pulse(row):
    if abs(row['pulse_end_h']) <= 5.0 / 3600.0:
        return 'inoculation (defines t=0; not nutrition)'
    if row['pulse_end_h'] < 0:
        return 'pre-inoculation / commissioning'
    if row['pulse_start_h'] >= 48.0:
        return 'possible later nutrition; dose unknown; not modeled'
    return 'unresolved post-inoculation process/sampling pulse; not modeled'

pulse_table['classification'] = pulse_table.apply(classify_pulse, axis=1)
for lab in LABS:
    lab_pulses = pulse_table[pulse_table['LAB'].eq(lab)]
    matches = lab_pulses['pulse_end'].eq(T0_BY_LAB[lab])
    assert matches.sum() == 1, f'{lab}: t0 does not match exactly one pump falling edge'
FIRST_POSSIBLE_NUTRITION_H = (
    pulse_table[pulse_table['classification'].str.startswith('possible later nutrition')]
    .groupby('LAB')['pulse_start_h'].min().to_dict()
)

def score_pulse_against_raw(row):
    raw = signals[row['LAB']]['co2_raw'].sort_values('timestamp')
    ts_ns = raw['timestamp'].astype('int64').to_numpy()
    values = raw['flow_sccm'].to_numpy(dtype=float)
    start_ns, end_ns = pd.Timestamp(row['pulse_start']).value, pd.Timestamp(row['pulse_end']).value
    p0, p1 = np.searchsorted(ts_ns, [start_ns - int(IIR_TAU_S * 1e9), start_ns])
    e0 = np.searchsorted(ts_ns, start_ns)
    e1 = np.searchsorted(ts_ns, end_ns + int(HAMPEL_WINDOW_S * 1e9), side='right')
    pre, event = values[p0:p1], values[e0:e1]
    baseline_span_min = (ts_ns[p1 - 1] - ts_ns[p0]) / 60e9 if len(pre) >= 2 else np.nan
    if len(pre) < 3 or len(event) == 0:
        return pd.Series({'baseline_raw_sccm': np.nan, 'baseline_sigma_sccm': np.nan, 'artifact_threshold_sccm': np.nan, 'peak_deviation_sccm': np.nan, 'event_score': np.nan, 'artifact_candidate': pd.NA, 'artifact_onset': pd.NaT, 'baseline_span_min': baseline_span_min})
    baseline = float(np.median(pre))
    sigma = float(1.4826 * np.median(np.abs(pre - baseline)))
    threshold = max(HAMPEL_K * sigma, HAMPEL_MIN_ABS_SCCM)
    deviation = np.abs(event - baseline)
    crossings = np.flatnonzero(deviation > threshold)
    detected = bool(len(crossings))
    onset = pd.to_datetime(ts_ns[e0 + crossings[0]]) if detected else pd.NaT
    return pd.Series({'baseline_raw_sccm': baseline, 'baseline_sigma_sccm': sigma, 'artifact_threshold_sccm': threshold, 'peak_deviation_sccm': float(deviation.max()), 'event_score': float(deviation.max() / threshold), 'artifact_candidate': detected, 'artifact_onset': onset, 'baseline_span_min': baseline_span_min})

pulse_scores = pd.concat([pulse_table, pulse_table.apply(score_pulse_against_raw, axis=1)], axis=1)
pulse_scores['artifact_candidate'] = pulse_scores['artifact_candidate'].astype('boolean')
eligible_reference = pulse_scores['artifact_candidate'].fillna(False) & pulse_scores['baseline_span_min'].ge(0.95 * IIR_TAU_S / 60.0)
reference_events = (pulse_scores.loc[eligible_reference].assign(baseline_abs=lambda d: d['baseline_raw_sccm'].abs()).sort_values(['LAB', 'baseline_abs', 'pulse_start']).groupby('LAB', group_keys=False).head(1))
assert set(reference_events['LAB']) == set(LABS)

def estimate_filtered_recovery(row):
    raw = signals[row['LAB']]['co2_raw'].sort_values('timestamp').copy()
    filt = signals[row['LAB']]['co2_filt'].sort_values('timestamp').copy()
    raw_dt_s = float(raw['timestamp'].diff().dt.total_seconds().median())
    filt_dt_s = float(filt['timestamp'].diff().dt.total_seconds().median())
    min_periods = max(3, int(round((IIR_TAU_S / raw_dt_s) / 4.0)))
    raw_trend = raw.set_index('timestamp')['flow_sccm'].rolling(f'{int(IIR_TAU_S)}s', center=True, min_periods=min_periods).median()
    trend_frame = raw_trend.rename('raw_robust_trend_sccm').dropna().reset_index()
    aligned = pd.merge_asof(filt, trend_frame, on='timestamp', direction='nearest', tolerance=pd.to_timedelta(raw_dt_s, unit='s')).dropna(subset=['raw_robust_trend_sccm'])
    aligned['filter_residual_sccm'] = aligned['flow_filt_sccm'] - aligned['raw_robust_trend_sccm']
    start, end = row['pulse_start'], row['pulse_end']
    pre = aligned.loc[aligned['timestamp'].between(start - pd.to_timedelta(IIR_TAU_S, unit='s'), start, inclusive='left'), 'filter_residual_sccm']
    center = float(pre.median())
    sigma = float(1.4826 * np.median(np.abs(pre - center)))
    tolerance = max(HAMPEL_K * sigma, HAMPEL_MIN_ABS_SCCM)
    post = aligned.loc[aligned['timestamp'].ge(end)].copy()
    post['compatible'] = post['filter_residual_sccm'].sub(center).abs().le(tolerance)
    sustain_n = max(1, int(round(IIR_TAU_S / filt_dt_s)))
    fraction = post['compatible'].rolling(sustain_n, min_periods=sustain_n).mean()
    endings = np.flatnonzero(fraction.to_numpy() >= 0.90)
    if not len(endings):
        return pd.Series({'recovery_start': pd.NaT, 'recovery_after_pulse_end_min': np.nan, 'recovery_tolerance_sccm': tolerance})
    recovery_i = max(0, int(endings[0]) - sustain_n + 1)
    recovery = post['timestamp'].iloc[recovery_i]
    return pd.Series({'recovery_start': recovery, 'recovery_after_pulse_end_min': (recovery - end).total_seconds() / 60.0, 'recovery_tolerance_sccm': tolerance})

recovery_diagnostics = pd.concat([reference_events.reset_index(drop=True), reference_events.apply(estimate_filtered_recovery, axis=1).reset_index(drop=True)], axis=1)
assert recovery_diagnostics['recovery_after_pulse_end_min'].notna().all()
COMMON_RECOVERY_MIN = int(np.ceil(recovery_diagnostics['recovery_after_pulse_end_min'].quantile(0.95)))
assert COMMON_RECOVERY_MIN == 90, f'Faithful mask audit no longer reproduces 90 min (got {COMMON_RECOVERY_MIN})'

artifact_table = pulse_scores.copy()
artifact_table['artifact_detected'] = artifact_table['artifact_candidate'].astype('boolean')
artifact_table['artifact_start'] = artifact_table['artifact_onset']
artifact_table['artifact_end'] = pd.NaT
detected = artifact_table['artifact_detected'].fillna(False)
artifact_table.loc[detected, 'artifact_end'] = artifact_table.loc[detected, 'pulse_end'] + pd.to_timedelta(COMMON_RECOVERY_MIN, unit='min')

def merge_windows(frame):
    merged = []
    for start, end in sorted(zip(frame['artifact_start'], frame['artifact_end'])):
        if not merged or start > merged[-1][1]:
            merged.append([start, end])
        else:
            merged[-1][1] = max(merged[-1][1], end)
    return [(start, end) for start, end in merged]

EXCLUSION_WINDOWS = {}
mask_audit_rows = []
for lab in LABS:
    co2 = signals[lab]['co2_filt']
    t0 = T0_BY_LAB[lab]
    co2['time_h'] = (co2['timestamp'] - t0).dt.total_seconds() / 3600.0
    signals[lab]['temp']['time_h'] = (signals[lab]['temp']['timestamp'] - t0).dt.total_seconds() / 3600.0
    lab_detected = artifact_table[artifact_table['LAB'].eq(lab) & artifact_table['artifact_detected'].fillna(False)]
    EXCLUSION_WINDOWS[lab] = merge_windows(lab_detected)
    co2['co2_valid_for_fitting'] = True
    for start, end in EXCLUSION_WINDOWS[lab]:
        co2.loc[co2['timestamp'].between(start, end, inclusive='both'), 'co2_valid_for_fitting'] = False
    mask_audit_rows.append({'LAB': lab, 'excluded_pct_all_record': 100.0 * (~co2['co2_valid_for_fitting']).sum() / len(co2), 'n_exclusion_windows': len(EXCLUSION_WINDOWS[lab])})
mask_audit = pd.DataFrame(mask_audit_rows)

t0_table = pd.DataFrame([{'LAB': lab, 'fermenter': LABS[lab]['fermenter'], 't0_inoculation': T0_BY_LAB[lab], 'verification': 'exactly matches pump falling edge', 'first_possible_nutrition_h': FIRST_POSSIBLE_NUTRITION_H[lab]} for lab in LABS])
display(t0_table)
display(pulse_table[['LAB', 'fermenter', 'pulse_start', 'pulse_end', 'pulse_start_h', 'pulse_end_h', 'duration_min', 'classification']].round(4))
print('Reference-event recovery audit; P95 rounded up =', COMMON_RECOVERY_MIN, 'min')
display(recovery_diagnostics[['LAB', 'pulse_start_h', 'duration_min', 'recovery_after_pulse_end_min', 'recovery_tolerance_sccm']].round(4))
print('Faithful all-record mask check (matches the prior ~3.57%, 2.47%, 3.36% audit):')
display(mask_audit.round(4))

In [ ]:
events_by_lab={}
for lab in COHORT1:
    ev=pulse_table[pulse_table.LAB.eq(lab)&pulse_table.classification.str.startswith('possible later nutrition')]
    events_by_lab[lab]=tuple((float(t),.14) for t in ev.sort_values('pulse_start_h').pulse_start_h)
    assert len(ev)==(2 if lab=='LAB017' else 1)

obs_parts=[];sensor_rows=[]
for lab in COHORT1:
    frame=signals[lab]['co2_filt'].copy()
    valid=frame.co2_valid_for_fitting & frame.time_h.ge(0)
    zero_support=frame[valid & frame.time_h.le(12)]
    zero=max(0.,float(zero_support.flow_filt_sccm.quantile(.1)))
    assert len(zero_support)>10
    frame['value']=np.maximum(frame.flow_filt_sccm-zero,0)*FACTOR
    frame['bin']=np.floor(frame.time_h).astype(int)
    agg=frame[valid].groupby('bin').agg(y=('value','median'),n=('value','count'),lo=('time_h','min'),hi=('time_h','max'))
    # Reject bins touching any mask, including partial intervals, and incomplete final bins.
    horizon=float(frame.time_h.max())
    keep=(agg.hi-agg.lo>=.75)&(agg.n>=2)&(agg.index+1<=horizon)
    for a,b in EXCLUSION_WINDOWS[lab]:
        a=(a-T0_BY_LAB[lab]).total_seconds()/3600
        b=(b-T0_BY_LAB[lab]).total_seconds()/3600
        keep &= ~((agg.index<=b)&(agg.index+1>=a))
    out=agg[keep].copy();out['t_h']=out.index+.5;out['lab']=lab
    obs_parts.append(out[['lab','t_h','y']])
    sensor_rows.append(dict(lab=lab,zero_sccm=zero,zero_support_n=len(zero_support),
        zero_first_h=zero_support.time_h.min(),zero_last_h=zero_support.time_h.max(),last_raw_h=horizon))

# Registros completos recientes, con corte en la última química de cada LAB.
recent_chemistry_metadata=pd.read_csv(PRIOR/'chemistry_samples_with_NUT.csv')
RECENT_END_H=recent_chemistry_metadata[~recent_chemistry_metadata.is_NUT].groupby('lab').t_h.max().to_dict()
recent_mask_path=LAB_DIR/'notebooks/EKF/post_activity_prediction_results/mask_audit.csv'
RECENT_MASK=pd.read_csv(recent_mask_path).set_index('lab')
recent_parts=[]
for chunk in pd.read_csv(RECENT_CO2_PATH,usecols=['timestamp','fermentador','flow_filt_sccm','status'],chunksize=300000):
    ts=pd.to_datetime(chunk.timestamp,errors='coerce')
    keep=ts.between(RECENT_T0,RECENT_T0+pd.Timedelta(hours=max(RECENT_END_H.values()))) & chunk.fermentador.isin(RECENT_FERMENTERS.values())
    if keep.any():
        d=chunk.loc[keep].copy();d['timestamp']=ts[keep]
        d['t_h']=(d.timestamp-RECENT_T0).dt.total_seconds()/3600.;recent_parts.append(d)
RECENT_RAW=pd.concat(recent_parts,ignore_index=True).sort_values(['fermentador','timestamp'])
RECENT_RAW['flow_filt_sccm']=pd.to_numeric(RECENT_RAW.flow_filt_sccm,errors='raise')
assert np.isfinite(RECENT_RAW.flow_filt_sccm).all()
recent_hourly=[];recent_sensor=[];recent_coverage=[]
previous_recent=pd.read_csv(OUT/'observations.csv') if (OUT/'observations.csv').exists() else None
for lab,fermenter in RECENT_FERMENTERS.items():
    horizon=float(RECENT_END_H[lab]);mask_end=float(RECENT_MASK.loc[lab,'mask_end_h'])
    d=RECENT_RAW[RECENT_RAW.fermentador.eq(fermenter)&RECENT_RAW.t_h.le(horizon)].copy()
    assert d.timestamp.is_unique, f'{lab}: duplicate gas timestamps'
    assert horizon-d.t_h.max()<1/60.,f'{lab}: complete gas file does not reach final chemistry'
    support=d[d.t_h.gt(mask_end)&d.t_h.le(12.)]
    zero=max(0.,float(support.flow_filt_sccm.quantile(.10)))
    assert np.isclose(zero,RECENT_MASK.loc[lab,'sensor_zero_sccm'],atol=1e-12)
    d['bin']=np.floor(d.t_h).astype(int)
    d['value']=np.maximum(d.flow_filt_sccm-zero,0.)*FACTOR
    hourly=d.groupby('bin').agg(y=('value','median'),n_raw=('value','size'),first_raw_h=('t_h','min'),last_raw_h=('t_h','max')).reset_index()
    hourly['bin_start_h']=hourly['bin'].astype(float);hourly['bin_end_h']=hourly.bin_start_h+1.
    hourly['t_h']=hourly.bin_start_h+.5;hourly['lab']=lab
    hourly['valid']=(hourly.bin_start_h.ge(mask_end)&hourly.bin_end_h.le(horizon)&
        hourly.n_raw.ge(2)&(hourly.last_raw_h-hourly.first_raw_h).ge(.75))
    assert hourly[hourly.valid].bin_end_h.max()==horizon
    recent_hourly.append(hourly)
    obs_parts.append(hourly[hourly.valid][['lab','t_h','y']])
    recent_sensor.append(dict(lab=lab,zero_sccm=zero,zero_support_n=len(support),
        zero_first_h=support.t_h.min(),zero_last_h=support.t_h.max(),mask_end_h=mask_end,
        last_raw_h=d.t_h.max(),last_hourly_bin_end_h=hourly[hourly.valid].bin_end_h.max(),
        chemistry_end_h=horizon,source=RECENT_CO2_PATH.name))
    old=previous_recent[previous_recent.lab.eq(lab)] if previous_recent is not None else pd.DataFrame()
    if len(old):
        overlap=old.merge(hourly[hourly.valid],on=['lab','t_h'],suffixes=('_previous','_complete'))
        overlap_error=float(np.max(abs(overlap.y_previous-overlap.y_complete)))
        assert overlap_error<1e-12
        previous_end=float(old.t_h.max()+.5)
    else:overlap_error=np.nan;previous_end=np.nan
    recent_coverage.append(dict(lab=lab,chemistry_end_h=horizon,gas_last_raw_h=d.t_h.max(),
        gas_last_bin_end_h=hourly[hourly.valid].bin_end_h.max(),previous_bin_end_h=previous_end,
        previous_overlap_max_error=overlap_error,valid_hourly_bins=int(hourly.valid.sum())))
RECENT_HOURLY=pd.concat(recent_hourly,ignore_index=True)
RECENT_COVERAGE=pd.DataFrame(recent_coverage)
OBS=pd.concat(obs_parts,ignore_index=True).sort_values(['lab','t_h']).reset_index(drop=True)
assert set(OBS.lab)==set(ALL_LABS)
SENSOR=pd.concat([pd.DataFrame(sensor_rows),pd.DataFrame(recent_sensor)],ignore_index=True)
display(SENSOR);display(RECENT_COVERAGE.round(6))
display(OBS.groupby('lab').agg(n=('y','size'),start_h=('t_h','min'),end_h=('t_h','max'),peak=('y','max')))
# Temperatura completa: se conservan puntos de soporte antes/después del horizonte para interpolar.
recent_temp_parts=[]
for chunk in pd.read_csv(RECENT_TEMP_PATH,usecols=['timestamp','fermentador','T'],chunksize=400000):
    ts=pd.to_datetime(chunk.timestamp,errors='coerce')
    keep=ts.between(RECENT_T0-pd.Timedelta(hours=1),RECENT_T0+pd.Timedelta(hours=max(RECENT_END_H.values())+1)) & chunk.fermentador.isin(RECENT_FERMENTERS.values())
    if keep.any():
        d=chunk.loc[keep].copy();d['timestamp']=ts[keep]
        d['t_h']=(d.timestamp-RECENT_T0).dt.total_seconds()/3600.;recent_temp_parts.append(d)
RECENT_TEMPERATURE=pd.concat(recent_temp_parts,ignore_index=True)
RECENT_TEMPERATURE['T']=pd.to_numeric(RECENT_TEMPERATURE['T'],errors='raise')
assert np.isfinite(RECENT_TEMPERATURE['T']).all()
for lab,fermenter in RECENT_FERMENTERS.items():
    tmp=RECENT_TEMPERATURE[RECENT_TEMPERATURE.fermentador.eq(fermenter)]
    assert tmp.t_h.min()<=0 and tmp.t_h.max()>=RECENT_END_H[lab]
protected_paths += [RECENT_CO2_PATH,RECENT_TEMP_PATH,recent_mask_path]


## 3. Condiciones iniciales y muestras corregidas de nitrógeno

LAB019–021 leen el YAN de `data/mem2026/LAB019-021/YAN_muestras_corregidas.csv` mediante una unión única por LAB y muestra. Se usan sus valores para los puntos experimentales y para N₀; sólo se convierten unidades de mg/L a g/L. **No se resta nada ni se trunca N en esta preparación ni en las ecuaciones del modelo.** La corrección de las muestras se hizo previamente en el CSV, conservando los exports originales del Y15.

El resto de la química y los metadatos proceden de la tabla anterior, sin recalcular el YAN corregido desde PAN/AMMONIA. Se comprueba que las 39 muestras, incluidas NUT, coincidan y que sus horarios sean los mismos. NUT se dibuja como contexto y se excluye de los errores porque su hora de extracción no está confirmada. Se conserva el G+F final de LAB019 igual a cero, acordado anteriormente.

La biomasa viable ponderada mantiene X₀=0.28316/0.15334/0.13615 g/L y Xd₀=0.032661 g/L. Las mediciones de biomasa de estos LAB no se utilizan para calibrar ni validar cuantitativamente.

LAB016–018 no están en el CSV corregido. Se usa la concentración viable inicial común de **13×10⁶ células/mL** indicada por el usuario, que da **X₀=13×0.03=0.39 g/L**. Se conservan N₀=0.21433 g/L, Xd₀=0, E₀=0 y las dosis de N acordadas. El N₀ se especifica directamente como condición inicial; no se realiza un descuento nuevo del Excel ni se afirma que sea una medición reciente.

Los recuentos Oculyze de LAB016–018 se leen de sus reportes originales y se comparan con X viable. Son muestras posteriores a la inoculación: no se adoptan como condiciones iniciales nuevas. Para LAB019–021 la temperatura se lee del archivo completo hasta la última química, en lugar de prolongar el último valor del registro anterior.


**Actualización de G+F inicial en LAB016–018:** el usuario reportó una densidad común de **1069 g/L a las 00:00 del 01/09/2026**. Con la relación de Lima, \(G_0+F_0=(1069-1000)/0.4=172.5\) g/L. Se usa esta muestra como condición inicial y se dibuja en t=0. El fin de inoculación registrado ocurre unos cuatro minutos después; se conserva el reloj del proceso y se documenta la correspondencia de esta muestra con t=0, sin desplazar las demás muestras.

La densidad sólo determina G+F. Para inicializar los dos estados, **se conserva como hipótesis la proporción anterior G:F = 74.05:77.73** y se escala su suma a 172.5 g/L; no se presentan G₀/F₀ como mediciones individuales. X₀, Xd₀, N₀, E₀, Gly₀, los pulsos y los parámetros químicos se mantienen. Se regeneran las trayectorias y los ajustes gaseosos porque cambió una entrada química.


El recuento inicial de 13×10⁶ células/mL ya corresponde a células **viables**: se convierte directamente a X₀, sin aplicar otro porcentaje de viabilidad. La conversión de 0.03 g/L por millón de células/mL es la misma del resto del notebook. Xd₀=0 sigue siendo la hipótesis previa; este cambio no proporciona un recuento inicial de células muertas. Se regeneran las predicciones y ajustes gaseosos con la nueva biomasa.


In [ ]:
chem_path=PRIOR/'chemistry_samples_with_NUT.csv'
inputs_path=PRIOR/'inputs.csv'
weighted_path=PRIOR/'inoculum_viable_weighted/initial_conditions.csv'
temp_manifest_path=PRIOR/'inoculum_viable_weighted/manifest.json'
temp_arrays_path=PRIOR/'inoculum_viable_weighted/trajectories.npz'
CHEM_ORIGINAL=pd.read_csv(chem_path)
CORRECTED_YAN=pd.read_csv(CORRECTED_YAN_PATH,dtype={'lab':str,'sample':str})
CHEM_ORIGINAL['sample']=CHEM_ORIGINAL['sample'].astype(str)
assert len(CORRECTED_YAN)==len(CHEM_ORIGINAL)==39
assert not CORRECTED_YAN.duplicated(['lab','sample']).any()
assert set(CORRECTED_YAN.lab)==set(COHORT2)
assert CORRECTED_YAN.YAN_mg_l.notna().all() and CORRECTED_YAN.YAN_mg_l.ge(0).all()
metadata_check=CHEM_ORIGINAL.merge(CORRECTED_YAN,on=['lab','sample'],suffixes=('_original','_file'),
    how='outer',validate='one_to_one',indicator=True)
assert metadata_check['_merge'].eq('both').all()
assert np.allclose(metadata_check.t_h_original,metadata_check.t_h_file)
assert metadata_check.is_NUT_original.equals(metadata_check.is_NUT_file)
assert metadata_check.time_provisional_original.equals(metadata_check.time_provisional_file)
CHEM=CHEM_ORIGINAL.drop(columns=['YAN_mg_l','N_kg_m3']).merge(
    CORRECTED_YAN[['lab','sample','YAN_mg_l']],on=['lab','sample'],how='left',validate='one_to_one')
CHEM['N_kg_m3']=CHEM.YAN_mg_l/1000.
CORRECTED_N0=CHEM[CHEM.is_initial].set_index('lab').N_kg_m3.to_dict()
assert set(CORRECTED_N0)==set(COHORT2)
N0_EXISTING_016_018=.21433  # Same prior initial-input hypothesis; no new sample correction.
INPUTS=pd.read_csv(inputs_path).set_index('lab')
WEIGHTED=pd.read_csv(weighted_path).set_index('lab')
TEMP_META=json.loads(temp_manifest_path.read_text(encoding='utf-8'))
TEMP_ARRAYS=np.load(temp_arrays_path)
INITIAL_VIABLE_016_018_MILLION_ML=13.
BIOMASS_G_L_PER_MILLION_ML=.03
INITIAL_X_016_018_G_L=INITIAL_VIABLE_016_018_MILLION_ML*BIOMASS_G_L_PER_MILLION_ML
assert np.isclose(INITIAL_X_016_018_G_L,.39,rtol=0,atol=1e-12)
INITIAL_DENSITY_016_018_G_L=1069.
INITIAL_GF_016_018_G_L=(INITIAL_DENSITY_016_018_G_L-1000.)/.4
assert np.isclose(INITIAL_GF_016_018_G_L,172.5)
BATCHES={};init_rows=[];INITIAL_SUGAR_ROWS=[]
for lab in ALL_LABS:
    if lab in COHORT1:
        row=initial_sheet[initial_sheet.CodigoLab.eq(lab)].iloc[0]
        ini=dict(X=INITIAL_X_016_018_G_L,Xd=0.,N=N0_EXISTING_016_018,G=float(row.Glucosa_g_L),
                 F=float(row.Fructosa_g_L),E=0.,Gly=float(row.Glicerol_g_L))
        previous_g,previous_f=ini['G'],ini['F']
        scale=INITIAL_GF_016_018_G_L/(previous_g+previous_f)
        ini['G']=previous_g*scale
        ini['F']=INITIAL_GF_016_018_G_L-ini['G']
        assert np.isclose(ini['G']/ini['F'],previous_g/previous_f,rtol=1e-12)
        assert np.isclose(ini['G']+ini['F'],172.5,rtol=0,atol=1e-12)
        measurement_timestamp=T0_BY_LAB[lab].normalize()  # User: 00:00 on inoculation day.
        INITIAL_SUGAR_ROWS.append(dict(lab=lab,sample='inicial',sample_timestamp=measurement_timestamp,
            sample_t_h=(measurement_timestamp-T0_BY_LAB[lab]).total_seconds()/3600.,t_h=0.,
            density_g_l=INITIAL_DENSITY_016_018_G_L,GF_density_g_l=INITIAL_GF_016_018_G_L,
            previous_G0_g_l=previous_g,previous_F0_g_l=previous_f,G0_g_l=ini['G'],F0_g_l=ini['F'],
            source='User-reported shared initial density at 00:00; assigned as initial condition',
            allocation='preserved previous G:F ratio; components not individually measured'))
        horizon=float(signals[lab]['co2_filt'].time_h.max())
        tmp=signals[lab]['temp'].sort_values('time_h')
        tx,ty=tmp.time_h.to_numpy(),tmp['T'].to_numpy()
        source='X0 por 13 millones de células viables/mL del usuario; G+F por densidad 1069; G:F proxy; Xd/E/N0/Gly y dosis N previas conservadas'
    else:
        row=INPUTS.loc[lab];ini={s:float(row[s]) for s in base.STATE_NAMES}
        ini.update(N=CORRECTED_N0[lab],X=float(WEIGHTED.loc[lab,'weighted_X0_g_l']),Xd=float(WEIGHTED.loc[lab,'new_common_Xd0_g_l']))
        events_by_lab[lab]=ast.literal_eval(row.N_pulses)
        tmp=RECENT_TEMPERATURE[RECENT_TEMPERATURE.fermentador.eq(RECENT_FERMENTERS[lab])].sort_values('t_h')
        tx,ty=tmp.t_h.to_numpy(),tmp['T'].to_numpy();horizon=float(RECENT_END_H[lab])
        source='N inicial m1 del CSV corregido; biomasa viable ponderada; temperatura completa hasta última química'
    assert ini['N']>=0
    events=events_by_lab[lab]
    grid=np.unique(np.r_[np.arange(0,horizon,.25),horizon,[t for t,d in events]])
    BATCHES[lab]=base.BatchData(medium='natural',batch=lab,time=grid,temperature_c=np.interp(grid,tx,ty),
        initials=ini,pulses={**{s:() for s in base.INPUT_CHANNELS},'N':events},
        observations={s:np.full(len(grid),np.nan) for s in base.STATE_NAMES})
    init_rows.append(dict(lab=lab,source=source,
        **ini,N_events=str(events),horizon_h=horizon))
INITIALS=pd.DataFrame(init_rows)
assert all(np.isclose(BATCHES[lab].initials['X'],.39,rtol=0,atol=1e-12) for lab in COHORT1)
INITIAL_SUGAR_016_018=pd.DataFrame(INITIAL_SUGAR_ROWS)
display(INITIALS)
display(Markdown('**Densidad inicial común reportada y reparto G:F asumido para LAB016–018.**'))
display(INITIAL_SUGAR_016_018[['lab','sample_timestamp','sample_t_h','density_g_l','GF_density_g_l','G0_g_l','F0_g_l']].round(6))
display(CHEM[['lab','sample','t_h','YAN_mg_l','N_kg_m3','is_NUT']])
assert all(BATCHES[lab].initials['N']==CORRECTED_N0[lab] for lab in COHORT2)
protected_paths += [CORRECTED_YAN_PATH,chem_path,inputs_path,weighted_path,temp_manifest_path,temp_arrays_path]
HASHES_BEFORE={str(p.relative_to(ROOT)):sha256(p) for p in set(protected_paths)}
# Recuentos Oculyze: no se desplazan a t=0 ni cambian las condiciones iniciales.
import re
BIOMASS_ROWS=[]
SPANISH_MONTHS={'enero':1,'febrero':2,'marzo':3,'abril':4,'mayo':5,'junio':6,
    'julio':7,'agosto':8,'septiembre':9,'octubre':10,'noviembre':11,'diciembre':12}
def oculyze_analysis_timestamp(row):
    match=re.fullmatch(r'(\d+) de (\w+) de (\d+)',str(row['Date']).strip())
    assert match, row['Date']
    day,month,year=match.groups()
    clock=str(row['Time']).lower().replace('a. m.','AM').replace('p. m.','PM').strip()
    stamp=pd.to_datetime(f'{year}-{SPANISH_MONTHS[month]:02d}-{int(day):02d} '+clock,format='%Y-%m-%d %I:%M:%S %p')
    return stamp
for lab in COHORT1:
    path=DATA_DIR/(lab+'_OCULYZE')/'report.csv'
    report=pd.read_csv(path)
    protected_paths.append(path)
    for _,r in report.iterrows():
        analysis=oculyze_analysis_timestamp(r)
        extraction=pd.to_datetime(str(r['Description']),dayfirst=True,errors='coerce')
        inconsistent=pd.isna(extraction) or extraction.date()!=analysis.date() or extraction>analysis
        if inconsistent:
            # An invalid extraction date is not silently repaired. Use the actual analysis timestamp.
            used=analysis;clock_source='analysis timestamp; inconsistent extraction description'
        else:used=extraction;clock_source='extraction description'
        total=float(r['Concentration']);viability=float(r['Viability'])/100.
        assert total>=0 and 0<=viability<=1
        BIOMASS_ROWS.append(dict(lab=lab,sample=str(r['Name']),sample_timestamp=used,
            analysis_timestamp=analysis,description=r['Description'],time_source=clock_source,
            extraction_time_inconsistent=bool(inconsistent),t_h=(used-T0_BY_LAB[lab]).total_seconds()/3600.,
            total_million_cells_ml=total,viability_fraction=viability,
            viable_million_cells_ml=total*viability,X_total_g_l=.03*total,
            X_viable_g_l=.03*total*viability,Xd_g_l=.03*total*(1-viability),source=str(path.relative_to(ROOT))))
BIOMASS_016_018=pd.DataFrame(BIOMASS_ROWS).sort_values(['lab','t_h']).reset_index(drop=True)
assert len(BIOMASS_016_018)==9 and BIOMASS_016_018.groupby('lab').size().eq(3).all()
assert BIOMASS_016_018.t_h.gt(0).all()
assert BIOMASS_016_018.extraction_time_inconsistent.sum()==1
display(Markdown('**Biomasa disponible de LAB016–018, en el horario de extracción indicado.** '
    'LAB018-1 tiene una fecha incompatible en Description; ese punto usa la hora real del análisis y queda identificado en la tabla. '
    'Conversión de recuentos a g/L: 0.03 g/L por millón de células/mL, igual que en el diagnóstico anterior. '
    'Se usa X₀=0.39 g/L, equivalente a los 13×10⁶ células viables/mL indicados por el usuario; Xd₀=0 permanece como hipótesis.'))
display(BIOMASS_016_018[['lab','sample','t_h','total_million_cells_ml','viable_million_cells_ml','X_viable_g_l','time_source']].round(5))
HASHES_BEFORE={str(p.relative_to(ROOT)):sha256(p) for p in set(protected_paths)}


## 4. Química fija: repetir simulaciones con las muestras corregidas de N

Se integra la misma cinética, con saltos exactos de nutrición y parámetros químicos congelados. Las dos variantes de theta pueden cambiar la captación de N, el consumo de azúcar y la producción de etanol. Dentro de cada variante, los ajustes Z1/Z2 de gas no cambian ninguna trayectoria química.

Los paneles G+F incorporan además los puntos calculados a partir de la densidad medida con la relación lineal de Lima, explicada a continuación.

### Contraste del azúcar con la densidad medida

En **Lima et al. (2024), sección 2.2, ecuación (2)** (página 5 del PDF local `context/EKF/Lima_2024.pdf`), se supone la relación lineal
\[
\rho=\frac{100}{250}S+1000=0.4S+1000,
\qquad S_{\rho}=\frac{\rho-1000}{0.4}.
\]
El artículo expresa \(\rho\) en kg/m³ y \(S\) en g/L. La densidad registrada aquí en g/L tiene el mismo valor numérico que en kg/m³. Comparamos ese azúcar total aproximado \(S_{\rho}\) con **G+F**, no con glucosa sola. La relación es una hipótesis de observación del artículo, donde también se supone temperatura constante; nuestros perfiles de temperatura varían. Su concordancia debe comprobarse en estos datos.

Los puntos verdes en los paneles G+F provienen de la **densidad medida**, aplicando esta ecuación directamente. En LAB016–018 se conservan las horas reales de extracción relativas a la inoculación; hay ocho muestras del Excel por LAB, hasta aproximadamente 56 h, más la densidad inicial común 1069 g/L reportada por el usuario, representada en t=0 y utilizada para inicializar G+F. En LAB019–021 se usa la densidad asociada a cada muestra y el mismo reloj de las figuras existentes (la primera muestra se representa en t=0). Los puntos negros de G+F siguen siendo los análisis del Y15.

La densidad inicial reportada sí actualiza G₀+F₀ de LAB016–018 a 172.5 g/L; se conservan las ecuaciones y parámetros químicos. Las densidades posteriores se muestran como comparación y no se usan para ajustar la química. No se modifican las muestras del Y15. **No se trunca el azúcar calculado por densidad**: si aparece negativo, indica que esta relación lineal devuelve un valor incompatible con una concentración física no negativa para esa muestra. No significa que exista azúcar negativo ni que debamos modificar el Y15 para hacerlo coincidir.

Las tablas siguientes muestran la cobertura y, para LAB019–021, la diferencia entre ambas estimaciones en las mismas muestras, sin interpolar. Un sesgo positivo significa que la densidad sobreestima G+F respecto del Y15. La concordancia entre estos métodos no valida por sí sola el modelo cinético.


In [ ]:
# Densidad: muestra inicial reportada usada en G+F, y muestras posteriores para comparar.
DENSITY_OUT=OUT/'density_sugar_comparison'
DENSITY_OUT.mkdir(exist_ok=True)
density_old=offline[['CodigoLab','Muestra','Fecha','Hora','Densidad_g_L']].copy()
density_old['sample_timestamp']=density_old.apply(lambda r:combine_date_time(r.Fecha,r.Hora),axis=1)
density_old['t_h']=(density_old.sample_timestamp-density_old.CodigoLab.map(T0_BY_LAB)).dt.total_seconds()/3600.
density_old=density_old.rename(columns={'CodigoLab':'lab','Muestra':'sample','Densidad_g_L':'density_g_l'})
density_old['GF_Y15_g_l']=np.nan
density_old['source']='LAB016-018-offline-measurements.xlsx / MedicionesOffline'
density_recent=CHEM_ORIGINAL[['lab','sample','sample_timestamp','sample_t_h','t_h','density','GF_g_l']].copy()
density_recent=density_recent.rename(columns={'density':'density_g_l','GF_g_l':'GF_Y15_g_l'})
density_recent['source']='chemistry_samples_with_NUT.csv / density'
density_initial=INITIAL_SUGAR_016_018[['lab','sample','sample_timestamp','sample_t_h','t_h','density_g_l','source']].copy()
density_initial['GF_Y15_g_l']=np.nan
DENSITY_POINTS=pd.concat([density_initial,density_old,density_recent],ignore_index=True)
for col in ['density_g_l','GF_Y15_g_l','t_h']:
    DENSITY_POINTS[col]=pd.to_numeric(DENSITY_POINTS[col],errors='raise')
DENSITY_POINTS=DENSITY_POINTS.dropna(subset=['density_g_l','t_h']).sort_values(['lab','t_h']).reset_index(drop=True)
DENSITY_POINTS['GF_density_g_l']=(DENSITY_POINTS.density_g_l-1000.)/.4
DENSITY_POINTS['density_minus_Y15_g_l']=DENSITY_POINTS.GF_density_g_l-DENSITY_POINTS.GF_Y15_g_l
DENSITY_POINTS['negative_density_sugar']=DENSITY_POINTS.GF_density_g_l.lt(0)
assert np.allclose(.4*DENSITY_POINTS.GF_density_g_l+1000.,DENSITY_POINTS.density_g_l)
assert set(DENSITY_POINTS.lab)==set(ALL_LABS)
assert DENSITY_POINTS.groupby('lab').size().loc[list(COHORT1)].eq(9).all()
DENSITY_COVERAGE=DENSITY_POINTS.groupby('lab',sort=True).agg(
    n_density=('density_g_l','size'),first_h=('t_h','min'),last_h=('t_h','max'),
    min_sugar_density_g_l=('GF_density_g_l','min'),negative_estimates=('negative_density_sugar','sum')).reset_index()
density_pairs=DENSITY_POINTS.dropna(subset=['GF_Y15_g_l'])
DENSITY_VS_Y15=pd.DataFrame([dict(lab=lab,n=len(d),bias_g_l=d.density_minus_Y15_g_l.mean(),
    RMSE_g_l=np.sqrt(np.mean(d.density_minus_Y15_g_l**2)),
    max_abs_difference_g_l=d.density_minus_Y15_g_l.abs().max()) for lab,d in density_pairs.groupby('lab',sort=True)])
display(DENSITY_COVERAGE.round(3))
display(DENSITY_VS_Y15.round(3))
for name,frame in [('density_sugar_points',DENSITY_POINTS),('density_coverage',DENSITY_COVERAGE),('density_vs_Y15',DENSITY_VS_Y15)]:
    frame.to_csv(DENSITY_OUT/(name+'.csv'),index=False)
density_manifest=dict(created_at=datetime.now().astimezone().isoformat(),
    relation='rho=0.4*S+1000; S_density=(rho-1000)/0.4',density_unit='g/L = kg/m3',sugar_unit='g/L',
    reference='Lima et al. 2024, section 2.2, equation (2), PDF page 5',
    paper_sha256=sha256(FM/'context/EKF/Lima_2024.pdf'),
    input_sha256={str(p.relative_to(ROOT)):sha256(p) for p in [WORKBOOK_PATH,PRIOR/'chemistry_samples_with_NUT.csv']},
    negative_estimates_clipped=False,initial_conditions_changed=True,chemical_parameters_refitted=False,gas_parameters_refitted=True,
    initial_density_report=INITIAL_SUGAR_016_018.assign(sample_timestamp=lambda d:d.sample_timestamp.astype(str)).to_dict('records'),
    recent_time_convention='existing sample t_h; m1 mapped to t=0',
    coverage=DENSITY_COVERAGE.to_dict('records'))
(DENSITY_OUT/'manifest.json').write_text(json.dumps(density_manifest,indent=2,ensure_ascii=False),encoding='utf-8')


In [ ]:
CORES={};CORES0={};CHEM_METRICS=[]
for tn,theta in THETAS.items():
    for lab,bt in BATCHES.items():
        CORES[tn,lab]=gas_model.integrate_core(bt,theta,bt.time)
        CORES0[tn,lab]=gas_model.integrate_core(replace(bt,pulses={**bt.pulses,'N':()}),theta,bt.time)
        core=CORES[tn,lab];core['GF']=core.G+core.F
        if lab in COHORT2:
            data=CHEM[CHEM.lab.eq(lab)&~CHEM.is_NUT & (CHEM.t_h>0)]
            for ch,col,mult in [('N','YAN_mg_l',1000.),('GF','GF_g_l',1.),('E','E_g_l',1.)]:
                pts=data[['t_h',col]].dropna()
                err=np.interp(pts.t_h,core.index,core[ch]*mult)-pts[col].to_numpy()
                CHEM_METRICS.append(dict(theta=tn,lab=lab,channel=ch,n=len(pts),RMSE=np.sqrt(np.mean(err**2)),bias=err.mean()))
CHEM_METRICS=pd.DataFrame(CHEM_METRICS)
display(CHEM_METRICS.round(4))
fig,axs=plt.subplots(3,4,figsize=(19,11),sharex=True)
for i,lab in enumerate(COHORT2):
    data=CHEM[CHEM.lab.eq(lab)]
    for j,(ch,col,mult,unit) in enumerate([('N','YAN_mg_l',1000.,'mg N/L corregido'),('X','X_viable_g_l',1.,'g/L'),('GF','GF_g_l',1.,'g/L'),('E','E_g_l',1.,'g/L')]):
        ax=axs[i,j]
        for tn,style,color in [('theta_full','-','#0072B2'),('theta_defaults','--','#D55E00')]:
            cr=CORES[tn,lab];ax.plot(cr.index,cr[ch]*mult,style,color=color,label=tn)
        d=data[~data.is_NUT].dropna(subset=[col])
        if ch=='X':ax.scatter(d.t_h,d[col],s=23,facecolors='none',edgecolors='#777777',label='Biomasa viable')
        else:ax.scatter(d.t_h,d[col],s=18,c='black',label='Medición')
        if ch=='GF':
            dp=DENSITY_POINTS[DENSITY_POINTS.lab.eq(lab)]
            ax.scatter(dp.t_h,dp.GF_density_g_l,s=23,c='#009E73',label='G+F por densidad',zorder=4)
        if ch=='N':
            d=data[data.is_NUT];ax.scatter(d.t_h,d[col],s=24,c='purple',label='NUT')
        for t,dose in events_by_lab[lab]:ax.axvline(t,c='deeppink',ls=':',lw=1)
        ax.set(title=f'{lab} · {ch}',ylabel=unit,xlim=(-3,float(BATCHES[lab].time[-1])+3));ax.grid(alpha=.3)
        if i==2:ax.set_xlabel('t desde inoculación [h]')
legend_items={}
for ax in axs[0]:
    hh,ll=ax.get_legend_handles_labels()
    for handle,label in zip(hh,ll):legend_items.setdefault(label,handle)
fig.legend(list(legend_items.values()),list(legend_items),loc='upper center',ncol=6)
fig.suptitle('Química con muestras corregidas de N y biomasa viable ponderada',y=1.02)
fig.tight_layout(rect=(0,0,1,.97));show(fig,'01_chemistry_recent')

### Química predicha de LAB016–018 que alimenta las transferencias

Se muestran N, biomasa viable X, G+F y etanol hasta el final de cada LAB, con `theta_full` y `theta_defaults`. Son **las mismas trayectorias químicas** utilizadas para construir la fuente de CO₂ de las figuras de transferencia; los parámetros gaseosos Z1/Z2 no modifican estas curvas.

G₀+F₀ se actualiza a **172.5 g/L**, calculado de la densidad inicial común **1069 g/L** reportada a las 00:00. Se conserva la proporción G:F = 74.05:77.73 para repartir el total; la tabla de entradas muestra ambos valores. Se usa **X₀=0.39 g/L**, convertido de **13×10⁶ células viables/mL** indicadas por el usuario. Se mantienen Xd₀=0, N₀=0.21433 g/L, E₀=0 y Gly₀=1.14 g/L. Se incorporan la temperatura registrada y los pulsos asumidos: 0.14 g N/L en LAB016/LAB018 y dos de 0.14 en LAB017. Las líneas verticales indican los horarios de esos pulsos; en LAB017 están muy próximos.

**No hay mediciones directas de N/G/F/E de esta corrida para contrastar estas curvas.** Se presentan como predicciones condicionadas a esas entradas, no como ajustes químicos validados. La figura permite ver qué evolución química presupone la capa de CO₂ y por qué el consumo predicho puede alcanzar 6 g/L temprano.

Los paneles G+F incluyen ahora el azúcar aproximado a partir de la densidad medida con la relación de Lima. Se dibujan nueve puntos por LAB: la muestra inicial común en t=0 y las ocho densidades posteriores del Excel hasta aproximadamente 56 h. Las predicciones se regeneran con G+F inicial de 172.5 g/L y X₀=0.39 g/L.

Los círculos de biomasa proceden de los reportes Oculyze, usando concentración total × viabilidad × 0.03. Son tres muestras por LAB y se sitúan después de t=0. La primera densidad del Excel es del 01/09 a las 09:00, unas 8.93 h después de la inoculación registrada cerca de las 00:04; ese Excel no contiene la densidad inicial, que ahora procede del dato aportado por el usuario. La muestra de las 00:00 se representa en t=0, unos cuatro minutos antes del fin de inoculación registrado, sin desplazar el resto de las mediciones. El reparto individual G₀/F₀ sigue siendo una hipótesis proporcional. Los horarios de biomasa provienen de Description cuando es consistente; para LAB018-1 (fecha 19/01/26 incompatible) se usa la hora real del análisis, sin inventar una extracción.


In [ ]:

fig,axs=plt.subplots(3,4,figsize=(19,11),sharex=True,sharey='col')
for i,lab in enumerate(COHORT1):
    for j,(channel,multiplier,unit) in enumerate([('N',1000.,'mg N/L'),('X',1.,'g/L'),('GF',1.,'g/L'),('E',1.,'g/L')]):
        ax=axs[i,j]
        for tn,color,style in [('theta_full','#0072B2','-'),('theta_defaults','#D55E00','--')]:
            cr=CORES[tn,lab]
            assert cr.index[0]==0. and cr.index[-1]>150.
            assert np.isfinite(cr[channel]).all() and cr[channel].min()>=0
            ax.plot(cr.index,cr[channel]*multiplier,c=color,ls=style,label=tn)
        if channel=='X':
            bp=BIOMASS_016_018[BIOMASS_016_018.lab.eq(lab)]
            ax.scatter(bp.t_h,bp.X_viable_g_l,s=23,facecolors='none',edgecolors='#777777',label='Biomasa viable',zorder=4)
        if channel=='GF':
            dp=DENSITY_POINTS[DENSITY_POINTS.lab.eq(lab)]
            ax.scatter(dp.t_h,dp.GF_density_g_l,s=23,c='#009E73',label='G+F por densidad',zorder=4)
        for k,(event,dose) in enumerate(events_by_lab[lab]):
            ax.axvline(event,c='deeppink',ls=':',lw=1.,label='Pulso N registrado' if k==0 else '_nolegend_')
        ax.set(title=f'{lab} · '+{'N':'N','X':'X','GF':'G+F','E':'Etanol'}[channel],ylabel=unit,xlim=(-3,float(BATCHES[lab].time[-1])+3))
        ax.grid(alpha=.3)
        if i==2:ax.set_xlabel('t desde inoculación [h]')
legend_items={}
for ax in axs[0]:
    hh,ll=ax.get_legend_handles_labels()
    for handle,label in zip(hh,ll):legend_items.setdefault(label,handle)
fig.legend(list(legend_items.values()),list(legend_items),ncol=5,loc='upper center')
fig.suptitle('Química predicha de LAB016–018 · X₀=0.39 g/L · densidad inicial 1069 g/L')
show(fig,'06_chemistry_lab016_018')


## 5. Cache de la fuente y verificación del balance

Sólo se optimiza la dinámica de liberación y su ganancia. Se precalculan la química, activación, transitorios y O₂ porque no dependen de Z. La implementación rápida del reservorio se contrasta numéricamente con `finish` antes y después de ajustar. Se comprueba el balance de CO₂; la ganancia de observación se aplica después del balance y no crea masa dentro del reservorio.

**Integración del reservorio:** el Euler explícito histórico puede oscilar cuando se ajusta una liberación rápida ($k\Delta t$ grande). Se conserva una implementación explícita sólo para auditar su reproducción de `finish`. Las curvas de calibración y comparación se calculan con Euler **implícito conservativo**, resolviendo la raíz positiva del balance en cada paso, que mantiene inventario y flujo no negativos sin oscilaciones de ese origen. Para $A=C_{d,i-1}+\Delta t\,q_{prod,i-1}$, se resuelve
$$ (1+k\Delta t)C_{d,i}^2+[C_{sat,i}(1+0.05k\Delta t)-A]C_{d,i}-A C_{sat,i}=0,$$
y $q_{sal,i}=(A-C_{d,i})/\Delta t$. Esto cambia el método numérico, no la ecuación física ni sus parámetros. Se verifica convergencia al subdividir los pasos, manteniendo constante la fuente en cada intervalo de la malla química.


En v2 se comprueba por LAB y theta la puerta progresiva de consumo de S=G+F entre 2 y 10 g/L. Los tiempos de inicio y activación completa se calculan después de simular para informar resultados; la puerta en cada instante usa sólo el consumo predicho de ese instante.


In [ ]:
def activation_from_sugar_consumption(sugar_consumed):
    z=np.clip((np.asarray(sugar_consumed,float)-SUGAR_ACTIVATION_START_G_L)/
        (SUGAR_ACTIVATION_FULL_G_L-SUGAR_ACTIVATION_START_G_L),0.,1.)
    return z*z*(3.-2.*z)

assert np.array_equal(activation_from_sugar_consumption([0.,2.,6.,10.,15.]),[0.,0.,.5,1.,1.])

def histories(time,events):
    ramp=np.clip((time-events[0][0])/OLD['pulse_t_rise_h'],0,1) if events else np.zeros(len(time))
    multiplier=np.ones(len(time))
    tr,td=OLD['tau_rise_h'],OLD['tau_decay_h']
    up=tr*np.log1p(td/tr);peak=(-np.expm1(-up/tr))*np.exp(-up/td)
    for event,dose in events:
        u=np.maximum(time-event,0.)
        multiplier += OLD['A']*dose/OLD['N_ref_g_l']*(-np.expm1(-u/tr))*np.exp(-u/td)/peak
    return ramp,multiplier

CACHE={};CHECKS=[];ACTIVATION_ROWS=[];ACTIVATION_PARTS=[]
for tn,theta in THETAS.items():
    for lab,bt in BATCHES.items():
        cr,cr0=CORES[tn,lab],CORES0[tn,lab]
        time=bt.time;ramp,multiplier=histories(time,bt.pulses['N'])
        sugar_consumed=cr.GF.iloc[0]-cr.GF.to_numpy()
        gate=activation_from_sugar_consumption(sugar_consumed)
        assert gate[0]==0. and np.all((gate>=0.)&(gate<=1.))
        assert np.all(np.diff(gate)>=-1e-12)
        assert np.all(gate[sugar_consumed<=SUGAR_ACTIVATION_START_G_L]==0.)
        assert np.all(gate[sugar_consumed>=SUGAR_ACTIVATION_FULL_G_L]==1.)
        active=np.flatnonzero(gate>0.);full=np.flatnonzero(gate==1.)
        assert len(active)>0 and len(full)>0
        first=int(active[0]);complete=int(full[0])
        assert 2.<sugar_consumed[first]<10. and sugar_consumed[complete]>=10.
        assert first>0 and gate[first-1]==0. and gate[complete-1]<1.
        for cutoff_h in (12.,30.,60.,100.):
            n=int(np.searchsorted(time,cutoff_h,side='right'))
            changed=sugar_consumed.copy();changed[n:]=-100.
            assert np.array_equal(activation_from_sugar_consumption(changed)[:n],gate[:n])
        ACTIVATION_ROWS.append(dict(theta=tn,lab=lab,S0_g_l=float(cr.GF.iloc[0]),
            first_partial_activation_h=float(time[first]),first_full_activation_h=float(time[complete]),
            consumed_at_first_partial_g_l=float(sugar_consumed[first]),
            consumed_at_first_full_g_l=float(sugar_consumed[complete]),
            gate_at_first_partial=float(gate[first]),
            ethanol_produced_at_first_partial_g_l=float(cr.E.iloc[first]-cr.E.iloc[0])))
        ACTIVATION_PARTS.append(pd.DataFrame(dict(t_h=time,lab=lab,theta=tn,
            sugar_consumed_g_l=sugar_consumed,gate=gate)))
        ini=bt.initials
        oxygen0=OLD['O2_initial_scale']*co2_cross.co2_model.o2_saturation_mg_l(
            bt.temperature_c[0],ini['E'],ini['G'],ini['F'],1.)
        ref=gas_model.finish(bt,theta,cr,cr0,OLD,gate,ramp,multiplier,0.,oxygen0)
        assert np.all(ref.qprod.to_numpy()[gate==0.] ==0.)
        assert np.all(ref.predicted.to_numpy()[:first+1]==0.)
        d=OBS[OBS.lab.eq(lab)].copy()
        assert d.t_h.max()<time[-1]
        idx=np.searchsorted(time,d.t_h.to_numpy())-1
        weight=(d.t_h.to_numpy()-time[idx])/(time[idx+1]-time[idx])
        CACHE[tn,lab]=dict(time=time,dt=np.diff(time),source=ref.qprod.to_numpy(),
            sat_base=ref.csat.to_numpy()/OLD['CO2sat_scale'],reference=ref,
            ramp=ramp,multiplier=multiplier,gate=gate,oxygen0=oxygen0,
            obs=d,idx=idx,w=weight)

def forward_explicit(c,params):
    k,s,g=[params[p] for p in ('kCO2_release_h','CO2sat_scale','matrix_gain')]
    source=c['source'];sat=c['sat_base']*s;dt=c['dt']
    cd=np.zeros(len(source));out=np.zeros(len(source))
    for i in range(1,len(source)):
        old=cd[i-1];step=dt[i-1]
        release=k*old*(.05+.95*old/max(old+sat[i],1e-12))
        out[i]=min(release,old/step+source[i-1])
        cd[i]=max(old+step*(source[i-1]-out[i]),0.)
    return g*out,cd,out


def forward(c,params,max_step=None):
    k,s,g=[params[p] for p in ('kCO2_release_h','CO2sat_scale','matrix_gain')]
    source=c['source'];sat=c['sat_base']*s;dt=c['dt']
    cd=np.zeros(len(source));out=np.zeros(len(source))
    for i in range(1,len(source)):
        subdivisions=1 if max_step is None else int(np.ceil(dt[i-1]/max_step))
        step=dt[i-1]/subdivisions;inventory=cd[i-1];released=0.
        for j in range(subdivisions):
            a=inventory+step*source[i-1]
            aa=1+step*k;b=sat[i]*(1+.05*step*k)-a
            root=np.sqrt(b*b+4*aa*a*sat[i])
            new_inventory=(2*a*sat[i]/(root+b) if b>=0 else (root-b)/(2*aa))
            released += a-new_inventory
            inventory=new_inventory
        cd[i]=inventory;out[i]=released/dt[i-1]
    return g*out,cd,out

def interpolate(c,pred):
    ix,w=c['idx'],c['w'];return pred[ix]*(1-w)+pred[ix+1]*w

for key,c in CACHE.items():
    pred,cd,qout=forward_explicit(c,OLD)
    err=float(np.max(np.abs(pred-c['reference'].predicted.to_numpy())))
    balance=float(np.max(np.abs(np.diff(cd)-c['dt']*(c['source'][:-1]-qout[1:]))))
    assert err<1e-12 and balance<1e-10
    CHECKS.append(dict(theta=key[0],lab=key[1],stage='initial',parity=err,balance=balance))
print('Cache contrastada con finish para las 12 trayectorias.')
ACTIVATION_TIMES=pd.DataFrame(ACTIVATION_ROWS)
ACTIVATION_TRAJECTORIES=pd.concat(ACTIVATION_PARTS,ignore_index=True)
display(Markdown('**Activación v2: consumo predicho de S=G+F, progresiva de 2 a 10 g/L.** '
    'Es nula hasta 2 g/L, vale 0.5 a 6 g/L y alcanza 1 a 10 g/L. '
    'La tabla informa los primeros nodos de activación parcial y completa en la malla de 0.25 h; '
    'no reinicializa ni recorta las trayectorias.'))
display(ACTIVATION_TIMES.round(4))


## 6. Función objetivo y calibración

Cada LAB contribuye con el mismo peso medio (1/n). Se utilizan todas sus horas válidas, incluidos inicio y cola. Para observaciones $y\ge0.05$ g/L/h, el residuo es $\hat y-y$; por debajo del límite de detección se penaliza sólo $\max(\hat y-0.05,0)$. Así, la cola no desaparece del ajuste ni se exige resolver valores menores que el límite. Los RMSE y sesgos que se presentan después **sí usan los valores observados completos**, sin censura.

Los parámetros se ajustan en logaritmos y dentro de los límites históricos. Z2 se inicia desde tres puntos, incluido Z1, y conserva el menor objetivo. Se exportan convergencia, límites y valores singulares locales: alcanzar un límite o tener una dirección débil advierte que un parámetro puede estar compensando errores estructurales. No se interpreta este diagnóstico como intervalo de confianza.

Z1 y Z2 se reajustan con la activación progresiva 2–10 g/L en todos los entrenamientos de v2. La referencia histórica se evalúa con esta misma activación y las mismas entradas.


In [ ]:
def residual(params,tn,labs):
    parts=[]
    for lab in labs:
        c=CACHE[tn,lab];p=interpolate(c,forward(c,params)[0]);y=c['obs'].y.to_numpy()
        e=np.where(y>=LOD,p-y,np.maximum(p-LOD,0.))
        parts.append(e/np.sqrt(len(e)))
    return np.concatenate(parts)

FIT_ROWS=[];MULTISTART=[];FITS={}
def fit(tn,labs,label):
    def obj(logg):return np.sum(residual({**OLD,'matrix_gain':np.exp(logg)},tn,labs)**2)
    scalar=minimize_scalar(obj,bounds=np.log(BOUNDS['matrix_gain']),method='bounded',options={'xatol':1e-9})
    z1={**OLD,'matrix_gain':float(np.exp(scalar.x))}
    names=list(BOUNDS);lo=np.log([BOUNDS[p][0] for p in names]);hi=np.log([BOUNDS[p][1] for p in names])
    seeds=[z1,{**z1,'kCO2_release_h':2.,'CO2sat_scale':1.},
        {**z1,'kCO2_release_h':10.,'CO2sat_scale':2.}]
    runs=[]
    for i,seed in enumerate(seeds):
        def fun(x):return residual({**OLD,**dict(zip(names,np.exp(x)))},tn,labs)
        sol=least_squares(fun,np.log([seed[p] for p in names]),bounds=(lo,hi),
            max_nfev=180,ftol=1e-8,xtol=1e-8,gtol=1e-8)
        runs.append(sol)
        MULTISTART.append(dict(theta=tn,training=label,seed=i,success=sol.success,nfev=sol.nfev,objective=np.sum(sol.fun**2),message=sol.message))
    best=min(runs,key=lambda x:np.sum(x.fun**2))
    z2={**OLD,**dict(zip(names,np.exp(best.x)))}
    assert np.sum(residual(z2,tn,labs)**2)<=np.sum(residual(z1,tn,labs)**2)+1e-7
    singular=np.linalg.svd(best.jac,compute_uv=False)
    for stage,p,success in [('Z1',z1,scalar.success),('Z2',z2,best.success)]:
        FITS[tn,label,stage]=p
        row=dict(theta=tn,training=label,stage=stage,labs=','.join(labs),success=bool(success),
            objective=float(np.sum(residual(p,tn,labs)**2)),
            jacobian_ratio=float(singular[-1]/singular[0]) if stage=='Z2' else np.nan)
        for name,(a,b) in BOUNDS.items():
            row[name]=p[name];row[name+'_bound']=(stage=='Z2' or name=='matrix_gain') and (abs(np.log(p[name]/a))<.005 or abs(np.log(p[name]/b))<.005)
        FIT_ROWS.append(row)
    print(tn,label,'Z1 / Z2:',round(obj(scalar.x),6),round(np.sum(best.fun**2),6),flush=True)

for tn in THETAS:
    fit(tn,ALL_LABS,'all_six')
    fit(tn,COHORT1,'train_016_018')
    fit(tn,COHORT2,'train_019_021')
FIT_TABLE=pd.DataFrame(FIT_ROWS)
display(FIT_TABLE.round(6))
display(pd.DataFrame(MULTISTART))

## 7. Resultados por LAB: inicio, máximo, caída y error

Se define inicio como tres horas válidas consecutivas por encima de 0.05 g/L/h; se informa la hora del último centro que confirma el evento (disponibilidad real: +0.5 h). La caída es el primer trío de horas válidas consecutivas por debajo del mismo límite después del máximo. Si no ocurre antes del fin del registro se deja ausente, sin inventar una fecha. El máximo corresponde a la serie horaria; los huecos enmascarados no se rellenan.

Las figuras y errores del ajuste conjunto describen el comportamiento dentro de calibración. La comprobación de transferencia se presenta aparte.

La referencia antigua de las figuras usa los parámetros gaseosos históricos, pero las mismas entradas actuales (N inicial del CSV corregido, biomasa ponderada y rendimiento corregido) y el mismo integrador que Z1/Z2. No es una reproducción literal de una figura previa con otras entradas.

In [ ]:
def features(t,y):
    t=np.asarray(t);y=np.asarray(y);peak=int(np.argmax(y))
    def run(mask,start=0):
        for i in range(max(2,start),len(t)):
            if mask[i-2:i+1].all() and np.allclose(np.diff(t[i-2:i+1]),1.):return float(t[i])
        return np.nan
    return dict(onset_h=run(y>=LOD),peak_h=float(t[peak]),peak=float(y[peak]),
                fall_h=run(y<LOD,peak+3))

METRICS=[];PREDICTIONS={};RESIDUALS=[]
for tn in THETAS:
    for lab in ALL_LABS:
        c=CACHE[tn,lab];d=c['obs'];t=d.t_h.to_numpy();y=d.y.to_numpy()
        observed=features(t,y)
        cases={'old_reference':OLD,**{stage:FITS[tn,'all_six',stage] for stage in ('Z1','Z2')}}
        for case,p in cases.items():
            pred,cd,qout=forward(c,p);sampled=interpolate(c,pred);err=sampled-y
            PREDICTIONS[tn,lab,case]=pd.DataFrame(dict(t_h=c['time'],CO2=pred,Cd=cd,qgas=qout,qprod=c['source']))
            f=features(t,sampled)
            row=dict(theta=tn,lab=lab,case=case,n=len(y),RMSE=np.sqrt(np.mean(err**2)),bias=err.mean(),
                **f,**{'observed_'+k:v for k,v in observed.items()})
            for k in ('onset_h','peak_h','fall_h','peak'):row['delta_'+k]=f[k]-observed[k]
            METRICS.append(row)
            RESIDUALS.extend(dict(theta=tn,lab=lab,case=case,t_h=a,y=b,predicted=c0,error=e) for a,b,c0,e in zip(t,y,sampled,err))
            balance=np.max(np.abs(np.diff(cd)-c['dt']*(c['source'][:-1]-qout[1:])))
            assert balance<1e-10
            if case=='Z2':
                bt=BATCHES[lab]
                ref=gas_model.finish(bt,THETAS[tn],CORES[tn,lab],CORES0[tn,lab],p,
                    c['gate'],c['ramp'],c['multiplier'],0.,c['oxygen0'])
                explicit=forward_explicit(c,p)[0]
                parity=np.max(np.abs(ref.predicted-explicit));assert parity<1e-12
                CHECKS.append(dict(theta=tn,lab=lab,stage=case,parity=parity,balance=balance))
METRICS=pd.DataFrame(METRICS);RESIDUALS=pd.DataFrame(RESIDUALS)
display(METRICS[['theta','lab','case','RMSE','bias','delta_onset_h','delta_peak_h','delta_peak','delta_fall_h']].round(4))
styles={'old_reference':('#999999',':','Referencia antigua'), 'Z1':('#E69F00','--','Z1 · ganancia'), 'Z2':('#0072B2','-','Z2 · liberación + solubilidad + ganancia')}
for tn in THETAS:
    fig,axs=plt.subplots(2,3,figsize=(17,9))
    for ax,lab in zip(axs.flat,ALL_LABS):
        d=CACHE[tn,lab]['obs'];ax.scatter(d.t_h,d.y,s=10,c='black',label='CO₂ observado',zorder=5)
        for case,(color,ls,label) in styles.items():
            pr=PREDICTIONS[tn,lab,case];ax.plot(pr.t_h,pr.CO2,color=color,ls=ls,label=label)
        for t,dose in events_by_lab[lab]:ax.axvline(t,c='deeppink',ls=':',lw=1)
        ax.set(title=lab,xlabel='t desde inoculación [h]',ylabel='g CO₂/L/h');ax.grid(alpha=.3)
    handles,labels=axs[0,0].get_legend_handles_labels();fig.legend(handles,labels,ncol=4,loc='upper center')
    fig.suptitle('Calibración con seis LAB · '+tn,y=1.02);fig.tight_layout(rect=(0,0,1,.96))
    show(fig,'02_gas_'+tn)
for tn in THETAS:
    fig,axs=plt.subplots(3,3,figsize=(16,11))
    for i,lab in enumerate(COHORT2):
        c=CACHE[tn,lab];d=c['obs'];obsf=features(d.t_h,d.y)
        windows=[(0,60),(max(0,obsf['peak_h']-25),obsf['peak_h']+25),(max(0,obsf['peak_h']+10),c['time'][-1])]
        for j,(left,right) in enumerate(windows):
            ax=axs[i,j];ax.scatter(d.t_h,d.y,s=12,c='black')
            for case,(color,ls,label) in styles.items():
                pr=PREDICTIONS[tn,lab,case];ax.plot(pr.t_h,pr.CO2,c=color,ls=ls,label=label)
            ax.axhline(LOD,c='#bbbbbb',ls=':',lw=1)
            ax.set(xlim=(left,right),title=lab+' · '+['Inicio','Máximo','Caída'][j],ylabel='g CO₂/L/h')
            ax.grid(alpha=.3)
            if i==2:ax.set_xlabel('t desde inoculación [h]')
    handles,labels=axs[0,0].get_legend_handles_labels();fig.legend(handles,labels,ncol=3,loc='upper center')
    fig.suptitle('Forma de la curva · '+tn,y=1.02);fig.tight_layout(rect=(0,0,1,.96))
    show(fig,'03_shape_'+tn)

## 8. Transferencia entre corridas

Dos comprobaciones separadas: ajustar LAB016–018 y predecir LAB019–021; ajustar LAB019–021 y predecir LAB016–018. Cada columna de parámetros se estima sin observar el CO₂ del grupo de evaluación. Se reporta el RMSE por LAB y la mejora respecto a la referencia antigua bajo las **mismas** entradas actuales. No es una validación con condiciones iniciales perfectamente medidas: las hipótesis de LAB016–018 condicionan tanto entrenamiento como evaluación.

Entrenamiento y predicción usan ambos la puerta progresiva v2 por consumo predicho de S=G+F entre 2 y 10 g/L.


In [ ]:
CV=[];CV_CURVES={}
for tn in THETAS:
    for training,test in [('train_016_018',COHORT2),('train_019_021',COHORT1)]:
        for stage in ('Z1','Z2'):
            p=FITS[tn,training,stage]
            for lab in test:
                c=CACHE[tn,lab];pred=forward(c,p)[0];sample=interpolate(c,pred);y=c['obs'].y.to_numpy()
                rmse=float(np.sqrt(np.mean((sample-y)**2)))
                old=METRICS[(METRICS.theta==tn)&(METRICS.lab==lab)&(METRICS.case=='old_reference')].iloc[0].RMSE
                CV.append(dict(theta=tn,training=training,stage=stage,test_lab=lab,RMSE=rmse,
                    bias=float(np.mean(sample-y)),improvement_pct=100*(1-rmse/old)))
                CV_CURVES[tn,training,stage,lab]=pred
CV=pd.DataFrame(CV);display(CV.round(4))
for tn in THETAS:
    fig,axs=plt.subplots(2,3,figsize=(17,9))
    for ax,lab in zip(axs.flat,ALL_LABS):
        training='train_019_021' if lab in COHORT1 else 'train_016_018'
        c=CACHE[tn,lab];d=c['obs'];ax.scatter(d.t_h,d.y,s=10,c='black',label='CO₂ observado')
        for stage,color,ls in [('Z1','#E69F00','--'),('Z2','#0072B2','-')]:
            ax.plot(c['time'],CV_CURVES[tn,training,stage,lab],c=color,ls=ls,label=stage+' · otra corrida')
        ax.set(title=f'{lab} · entrenado en '+('019–021' if lab in COHORT1 else '016–018'),
            xlabel='t desde inoculación [h]',ylabel='g CO₂/L/h');ax.grid(alpha=.3)
    handles,labels=axs[0,0].get_legend_handles_labels();fig.legend(handles,labels,ncol=3,loc='upper center')
    fig.suptitle('Transferencia entre corridas · '+tn,y=1.02);fig.tight_layout(rect=(0,0,1,.96))
    show(fig,'04_transfer_'+tn)

## 9. Qué puede corregir Z y qué queda fuera

Una ganancia reduce la amplitud pero no cambia el momento del máximo ni la relación entre las distintas partes de la curva. Aumentar $k$ vacía antes el reservorio; reducir $s$ aumenta la liberación a una misma concentración. Si la fuente $q_{prod}$ sigue activa cuando el caudal medido ya cayó, liberar más rápido no elimina esa fuente: queda un error estructural o de entradas/medición.

Se muestran fuente, salida sin ganancia y caudal observado para separar estos efectos. No se equiparan directamente sus amplitudes porque el observado se relaciona con la salida mediante $g$. El balance verifica el reservorio, pero no elimina la incoherencia potencial entre $q_{bio}$ y el etanol químico después del pulso: cuando $r=1$, la fuente incluye $q_n-0.75q_0$, mientras E sigue su propia trayectoria. Esto impide atribuir cualquier ganancia ajustada sólo al sensor.

La selección de theta debe sustentarse en química independiente y capacidad de transferencia, no sólo en el mejor CO₂ dentro del ajuste. El remanente de N no desacopla por sí solo la captación de N y la fermentación ni demuestra que LAB021 sea un problema exclusivamente térmico.

In [ ]:
for tn in THETAS:
    fig,axs=plt.subplots(1,3,figsize=(17,4.7))
    for ax,lab in zip(axs,COHORT2):
        pr=PREDICTIONS[tn,lab,'Z2'];c=CACHE[tn,lab]
        ax.plot(pr.t_h,pr.qprod,c='#D55E00',ls='--',label='Fuente qprod')
        ax.plot(pr.t_h,pr.qgas,c='#009E73',label='Salida qgas sin ganancia')
        ax.scatter(c['obs'].t_h,c['obs'].y,s=10,c='black',label='CO₂ observado')
        ax.set(title=lab,xlabel='t desde inoculación [h]',ylabel='g CO₂/L/h');ax.grid(alpha=.3)
    h,l=axs[0].get_legend_handles_labels();fig.legend(h,l,ncol=3,loc='upper center')
    fig.suptitle('Fuente frente a liberación · '+tn,y=1.04);fig.tight_layout(rect=(0,0,1,.94))
    show(fig,'05_source_'+tn)

display(Markdown('### Lectura cuantitativa del ajuste'))
summary=METRICS.groupby(['theta','case']).RMSE.mean().unstack('case')
display(summary.round(5))
for tn in THETAS:
    row=FIT_TABLE[(FIT_TABLE.theta==tn)&(FIT_TABLE.training=='all_six')&(FIT_TABLE.stage=='Z2')].iloc[0]
    bounds=[p for p in BOUNDS if row[p+'_bound']]
    display(Markdown(f'**{tn}**: k={row.kCO2_release_h:.4g} h⁻¹, s={row.CO2sat_scale:.4g}, '
        f'g={row.matrix_gain:.4g}. Límites activos: {", ".join(bounds) or "ninguno"}. '
        f'Cociente singular mínimo/máximo local: {row.jacobian_ratio:.3g}.'))
    transfer=CV[CV.theta.eq(tn)].groupby(['training','stage']).RMSE.mean()
    display(transfer.to_frame('RMSE medio en otra corrida'))

### Comprobaciones independientes

La refinación de malla comprueba la liberación, sin reestimar parámetros ni modificar la química. Se contrasta la solución implícita original con pasos de 0.125 y 0.0625 h, para los ajustes conjuntos y los dos grupos. También se comprueba el inventario $N+(q_N/\mu_0)(X+X_d)-\sum d_N$ con el convenio de muestra antes del pulso. Las observaciones recientes se reconstruyen otra vez desde el backup y su cero auditado, no sólo desde el CSV de resultados.


In [ ]:

REFINEMENT=[]
for (tn,training,stage),p0 in FITS.items():
    for lab in ALL_LABS:
        c=CACHE[tn,lab]
        a=interpolate(c,forward(c,p0)[0])
        b=interpolate(c,forward(c,p0,.125)[0])
        fine=interpolate(c,forward(c,p0,.0625)[0])
        REFINEMENT.append(dict(theta=tn,training=training,stage=stage,lab=lab,
            max_coarse_fine=float(np.max(np.abs(a-fine))),
            rms_coarse_fine=float(np.sqrt(np.mean((a-fine)**2))),
            rms_half_fine=float(np.sqrt(np.mean((b-fine)**2)))))
REFINEMENT=pd.DataFrame(REFINEMENT)
assert REFINEMENT.rms_coarse_fine.max()<.01, REFINEMENT.rms_coarse_fine.max()
assert REFINEMENT.rms_half_fine.max()<.005
display(REFINEMENT.groupby(['theta','training','stage'])[['rms_coarse_fine','rms_half_fine']].max())
N_BALANCE=[]
for (tn,lab),cr in CORES.items():
    doses=np.array([sum(d for t,d in events_by_lab[lab] if t<ti) for ti in cr.index])
    inventory=cr.N.to_numpy()+THETAS[tn]['qN']/THETAS[tn]['mu0']*(cr.X+cr.Xd).to_numpy()-doses
    error=float(np.max(np.abs(inventory-inventory[0])))
    assert error<2e-6 and cr[list(base.STATE_NAMES)].min().min()>=0
    N_BALANCE.append(dict(theta=tn,lab=lab,max_error_g_l=error))
N_BALANCE=pd.DataFrame(N_BALANCE)

# Independent reconstruction of the recent sensor observations from the raw backup.
rawpath=RECENT_CO2_PATH
parts=[];t0=pd.Timestamp('2026-09-21 17:00:00')
for chunk in pd.read_csv(rawpath,usecols=['timestamp','fermentador','flow_filt_sccm'],chunksize=300000):
    ts=pd.to_datetime(chunk.timestamp)
    keep=ts.between(t0,t0+pd.Timedelta(hours=max(RECENT_END_H.values())))
    if keep.any():
        ch=chunk[keep].copy();ch['t_h']=(ts[keep]-t0).dt.total_seconds()/3600;parts.append(ch)
raw=pd.concat(parts,ignore_index=True)
RECENT_AUDIT=[]
for lab,fermenter in zip(COHORT2,['F1','F2','F3']):
    f=raw[raw.fermentador.eq(fermenter)&raw.t_h.le(RECENT_END_H[lab])].copy()
    zero=float(SENSOR[SENSOR.lab.eq(lab)].zero_sccm.iloc[0])
    f['bin']=np.floor(f.t_h).astype(int)
    f['value']=np.maximum(pd.to_numeric(f.flow_filt_sccm)-zero,0)*FACTOR
    grouped=f.groupby('bin').value.median()
    d=OBS[OBS.lab.eq(lab)]
    rebuilt=grouped.reindex(np.floor(d.t_h).astype(int)).to_numpy()
    error=float(np.max(np.abs(rebuilt-d.y.to_numpy())))
    assert error<1e-12
    RECENT_AUDIT.append(dict(lab=lab,n=len(d),max_error_g_l_h=error))
RECENT_AUDIT=pd.DataFrame(RECENT_AUDIT)
del raw,parts
display(N_BALANCE);display(RECENT_AUDIT)


### Conclusiones y alcance de la activación v2

**Entradas y química:** LAB016–018 se simulan ahora con 13×10⁶ células viables/mL (X₀=0.39 g/L) y G₀+F₀=172.5 g/L por la densidad inicial 1069. Los parámetros químicos permanecen fijos; las curvas de esos LAB y los ajustes gaseosos se regeneraron con estas entradas. LAB019–021 conservan su química e inicialización anteriores. En ellos, `theta_full` sigue siendo preferible a `theta_defaults` para G+F y E, aunque en LAB021 conserva demasiado azúcar y produce etanol demasiado tarde en la zona central. LAB019 mantiene un error importante de captación temprana de N. Los recuentos Oculyze y las densidades posteriores de LAB016–018 aportan un contraste parcial; no equivalen a una validación directa de N, G/F y E.

**Activación y forma:** se mantiene la química del respaldo y se usa exclusivamente la transición progresiva por consumo predicho de S=G+F entre 2 y 10 g/L. Los tiempos de activación parcial/completa y las tablas de inicio, máximo, caída y errores muestran el comportamiento con esta regla. La ganancia y liberación se reajustan; por ello, las diferencias gaseosas respecto a v1 incluyen el cambio de activación y el reajuste de Z. El ajuste con seis LAB es calibración; las transferencias evalúan la otra corrida.

**Liberación y solubilidad:** los parámetros gaseosos se reajustan con X₀=0.39 g/L en LAB016–018. Sus valores y los indicadores de límites activos se muestran en las tablas de esta ejecución. Dependen de la química e inicialización utilizadas y no pueden interpretarse automáticamente como constantes físicas identificadas. Las transferencias entre corridas siguen siendo necesarias para evaluar su capacidad predictiva.

**Comparación Z1/Z2:** minimizar la función objetivo conjunta no garantiza mejorar cada LAB por separado, ni recuperar simultáneamente el pico y la cola. El resumen calculado debajo informa para cada LAB si Z2 aumenta o reduce el RMSE respecto a Z1, y qué parámetros alcanzan límites. Así las conclusiones numéricas corresponden a la inicialización actual.

**Siguiente camino:** conservar `theta_full` como base química de comparación y Z1 como referencia gaseosa sencilla. Revisar la fuente postnutrición (qₙ−0.75q₀) y su coherencia con E, y después evaluar una descripción de actividad fermentativa/temperatura que mejore la conversión central y el final sin alterar arbitrariamente el rendimiento. Contrastar cualquier cambio con N, G+F, E y CO₂ hasta el final, reservando una corrida para transferencia. Este notebook no demuestra todavía qué modificación estructural es la correcta y no modifica el modelo compartido.

Los parámetros de O₂ y respuesta a nutrición permanecen heredados y fijos; se reajustan únicamente los componentes libres de Z1/Z2. No se ha recalibrado toda la fisiología.

La activación usa el consumo de azúcar respecto a S₀ y no exige un umbral de etanol. Sigue siendo causal en la predicción y no necesita una muestra nueva en el instante de encendido.


In [ ]:

display(Markdown('**Resumen del ajuste conjunto — valores nuevos**'))
display(FIT_TABLE[FIT_TABLE.training.eq('all_six')][['theta','stage','kCO2_release_h','CO2sat_scale','matrix_gain','objective']].round(6))
display(Markdown('**LAB021 — máximo, caída y error completo**'))
display(METRICS[METRICS.lab.eq('LAB021')][['theta','case','RMSE','peak','observed_peak','fall_h','observed_fall_h']].round(4))
print('Máxima diferencia RMS por refinación del reservorio [g/L/h]:',REFINEMENT.rms_coarse_fine.max())

display(Markdown('**Comparación calculada para la inicialización actual — theta_full, ajuste conjunto.**'))
for lab in COHORT2:
    current=METRICS[METRICS.theta.eq('theta_full')&METRICS.lab.eq(lab)].set_index('case')
    rmse1=float(current.loc['Z1','RMSE']);rmse2=float(current.loc['Z2','RMSE'])
    direction='reduce' if rmse2<rmse1 else 'aumenta' if rmse2>rmse1 else 'mantiene'
    display(Markdown(f'**{lab}:** Z2 {direction} el RMSE de {rmse1:.4f} a {rmse2:.4f} g CO₂/L/h. '
        f'Máximo observado: {float(current.loc["Z2","observed_peak"]):.3f}; '
        f'máximo predicho Z2: {float(current.loc["Z2","peak"]):.3f} g CO₂/L/h.'))
for tn in THETAS:
    fitted=FIT_TABLE[FIT_TABLE.theta.eq(tn)&FIT_TABLE.training.eq('all_six')&FIT_TABLE.stage.eq('Z2')].iloc[0]
    active=[name for name in BOUNDS if bool(fitted[name+'_bound'])]
    display(Markdown(f'**{tn}, Z2 conjunto:** parámetros en límite: '+(', '.join(active) if active else 'ninguno')+'.'))


## 9.1. Prueba controlada de sF y sG: química y final del CO₂

Comparamos cinco casos: `theta_full`, sustituir únicamente `sF`, únicamente `sG`, ambos por sus valores de `theta_defaults`, y `theta_defaults` completo. **Son sustituciones diagnósticas, no una nueva calibración ni una selección de parámetros definitivos.**

En las ecuaciones actuales, $K_G(T)=\beta_{G,0}a_{K_G}(T)/s_G$ y $K_F(T)=\beta_{F,0}a_{K_F}(T)/s_F$ entran en $G/(G+K_G)$ y $F/(F+K_F)$. Los factores térmicos están normalizados a 20 °C. Al disminuir `sF` o `sG` aumenta la constante correspondiente y se reduce la captación fermentativa, especialmente cuando queda poco azúcar. Los mismos factores también participan en la producción de etanol. La prueba muestra ese acoplamiento; no evalúa solamente una demora del CO₂.

Se conservan las condiciones iniciales actuales, biomasa viable ponderada de LAB019–021, muestras YAN ya corregidas, temperaturas medidas, pulsos y horizonte completo. Se recalculan tanto la química con nutrición como la trayectoria sin nutrición que utiliza la fuente gaseosa, la saturación de CO₂ y la activación progresiva por consumo predicho de G+F entre 2 y 10 g/L. No se reinicia el estado ni se desplaza el reloj.

**Para aislar el efecto químico, todos los casos usan exactamente la misma capa gaseosa de `theta_full`.** La figura global usa su Z2 ajustado con seis LAB. La figura de transferencia usa su Z2 entrenado en la otra corrida. La curva `theta_defaults` de esta sección también usa esos parámetros gaseosos: por eso no coincide necesariamente con las figuras anteriores, donde tenía su propio ajuste. No se ejecuta ningún optimizador para estas sustituciones.


In [ ]:

AFFINITY_OUT=OUT/'sugar_affinity_parameter_comparison'
AFFINITY_OUT.mkdir(exist_ok=True)
AFFINITY_CASES={
    'full':THETAS['theta_full'].copy(),
    'full_sF_default':{**THETAS['theta_full'],'sF':THETAS['theta_defaults']['sF']},
    'full_sG_default':{**THETAS['theta_full'],'sG':THETAS['theta_defaults']['sG']},
    'full_sF_sG_default':{**THETAS['theta_full'],**{p:THETAS['theta_defaults'][p] for p in ('sF','sG')}},
    'defaults':THETAS['theta_defaults'].copy(),
}
AFFINITY_STYLES={
    'full':('theta_full','#0072B2','-'),
    'full_sF_default':('full · sF default','#009E73','--'),
    'full_sG_default':('full · sG default','#CC79A7','-.'),
    'full_sF_sG_default':('full · sF y sG default','#D55E00','-'),
    'defaults':('theta_defaults','#777777',':'),
}
AFFINITY_PARAMETERS=pd.DataFrame(AFFINITY_CASES)
AFFINITY_FIXED_GAS={label:FITS['theta_full',label,'Z2'].copy()
    for label in ('all_six','train_016_018','train_019_021')}
AFFINITY_GAS_SNAPSHOT=json.loads(json.dumps(AFFINITY_FIXED_GAS))
for case,changed in [('full_sF_default',{'sF'}),('full_sG_default',{'sG'}),('full_sF_sG_default',{'sF','sG'})]:
    assert {p for p in base.FULL17 if AFFINITY_CASES[case][p]!=THETAS['theta_full'][p]}==changed
AFFINITY_CORES={};AFFINITY_GAS={};AFFINITY_ROWS=[];AFFINITY_CHEM_ROWS=[]
AFFINITY_CHECKS=[];AFFINITY_TRAJECTORIES=[]
for lab,bt in BATCHES.items():
    obs=OBS[OBS.lab.eq(lab)].sort_values('t_h')
    ot,oy=obs.t_h.to_numpy(),obs.y.to_numpy()
    observed=features(ot,oy)
    after_peak=ot>=observed['peak_h']
    time=bt.time;ramp,multiplier=histories(time,bt.pulses['N'])
    ini=bt.initials
    oxygen0=OLD['O2_initial_scale']*co2_cross.co2_model.o2_saturation_mg_l(
        bt.temperature_c[0],ini['E'],ini['G'],ini['F'],1.)
    training_for_transfer='train_019_021' if lab in COHORT1 else 'train_016_018'
    print('Prueba sF/sG:',lab,flush=True)
    for case,theta in AFFINITY_CASES.items():
        original={'full':'theta_full','defaults':'theta_defaults'}.get(case)
        if original:
            cr=CORES[original,lab].copy();cr0=CORES0[original,lab].copy()
        else:
            cr=gas_model.integrate_core(bt,theta,time)
            cr0=gas_model.integrate_core(replace(bt,pulses={**bt.pulses,'N':()}),theta,time)
        cr['GF']=cr.G+cr.F
        assert cr.index[0]==0. and cr.index[-1]==bt.time[-1]
        assert np.isfinite(cr[list(base.STATE_NAMES)].to_numpy()).all()
        assert np.allclose(cr.iloc[0][list(base.STATE_NAMES)],pd.Series(ini)[list(base.STATE_NAMES)],atol=1e-10)
        AFFINITY_CORES[case,lab]=cr
        consumed=cr.GF.iloc[0]-cr.GF.to_numpy()
        gate=activation_from_sugar_consumption(consumed)
        assert np.all(gate[consumed<=2.] ==0.) and np.all(gate[consumed>=10.] ==1.)
        below=np.flatnonzero(cr.GF.to_numpy()<=5.)
        first_full=np.flatnonzero(gate==1.)
        for mode,training in [('global','all_six'),('transfer',training_for_transfer)]:
            params=AFFINITY_FIXED_GAS[training]
            ref=gas_model.finish(bt,theta,cr,cr0,params,gate,ramp,multiplier,0.,oxygen0)
            c=dict(time=time,dt=np.diff(time),source=ref.qprod.to_numpy(),
                sat_base=ref.csat.to_numpy()/params['CO2sat_scale'])
            pred,cd,qout=forward(c,params)
            balance=float(np.max(abs(np.diff(cd)-c['dt']*(c['source'][:-1]-qout[1:]))))
            assert balance<1e-10 and np.min(cd)>=0. and np.min(pred)>=-1e-12
            assert np.all(c['source'][gate==0.] ==0.)
            baseline_error=np.nan
            if case=='full':
                if mode=='global':
                    previous=pd.read_csv(OUT/'gas_trajectories.csv')
                    previous=previous[previous.theta.eq('theta_full')&previous.lab.eq(lab)&previous.case.eq('Z2')]
                    assert np.allclose(previous.t_h,time,atol=1e-12)
                    baseline_error=float(np.max(abs(previous.CO2.to_numpy()-pred)))
                else:
                    baseline_error=float(np.max(abs(forward(CACHE['theta_full',lab],params)[0]-pred)))
                assert baseline_error<1e-8,(lab,mode,baseline_error)
            sampled=np.interp(ot,time,pred);error=sampled-oy
            f=features(ot,sampled)
            row=dict(lab=lab,case=case,mode=mode,gas_training=training,
                RMSE_CO2=float(np.sqrt(np.mean(error**2))),
                RMSE_CO2_after_observed_peak=float(np.sqrt(np.mean(error[after_peak]**2))),
                G_end_g_l=float(cr.G.iloc[-1]),F_end_g_l=float(cr.F.iloc[-1]),
                GF_end_g_l=float(cr.GF.iloc[-1]),E_end_g_l=float(cr.E.iloc[-1]),
                first_GF_le_5_h=float(time[below[0]]) if len(below) else np.nan,
                first_gate_1_h=float(time[first_full[0]]) if len(first_full) else np.nan,
                horizon_h=float(time[-1]),**f,**{'observed_'+k:v for k,v in observed.items()})
            for k in ('onset_h','peak_h','peak','fall_h'):row['delta_'+k]=f[k]-observed[k]
            AFFINITY_ROWS.append(row)
            AFFINITY_CHECKS.append(dict(lab=lab,case=case,mode=mode,
                balance_error=balance,baseline_max_error=baseline_error))
            pr=pd.DataFrame(dict(t_h=time,CO2=pred,Cd=cd,qgas=qout,qprod=c['source'],gate=gate,
                sugar_consumed_g_l=consumed,lab=lab,case=case,mode=mode))
            AFFINITY_GAS[mode,case,lab]=pr
            AFFINITY_TRAJECTORIES.append(pr)
        if lab in COHORT2:
            data=CHEM[CHEM.lab.eq(lab)&~CHEM.is_NUT&CHEM.t_h.gt(0.)]
            for channel,col,mult in [('N','YAN_mg_l',1000.),('G','G_g_l',1.),('F','F_g_l',1.),('GF','GF_g_l',1.),('E','E_g_l',1.)]:
                d=data[['t_h',col]].dropna()
                if len(d):
                    err=np.interp(d.t_h,time,cr[channel]*mult)-d[col].to_numpy()
                    AFFINITY_CHEM_ROWS.append(dict(lab=lab,case=case,channel=channel,n=len(d),
                        RMSE=float(np.sqrt(np.mean(err**2))),bias=float(np.mean(err))))
AFFINITY_METRICS=pd.DataFrame(AFFINITY_ROWS)
AFFINITY_CHEM_METRICS=pd.DataFrame(AFFINITY_CHEM_ROWS)
assert AFFINITY_FIXED_GAS==AFFINITY_GAS_SNAPSHOT and THETAS==THETA_SNAPSHOT
display(AFFINITY_PARAMETERS.loc[['sF','sG','m0','sN','qXG','qXF']])
display(pd.DataFrame(AFFINITY_FIXED_GAS).loc[list(BOUNDS)])
display(Markdown('**CO₂: inicio, máximo y caída en la misma malla horaria de las mediciones.** '
    'La caída requiere tres observaciones horarias consecutivas bajo 0.05 g/L/h después del máximo. '
    'NaN indica que no se observa ese cruce dentro del horizonte. '
    'El RMSE posterior al máximo usa desde el máximo medido hasta el final.'))
display(AFFINITY_METRICS[['lab','case','mode','RMSE_CO2','RMSE_CO2_after_observed_peak',
    'delta_onset_h','delta_peak_h','delta_peak','delta_fall_h','GF_end_g_l','E_end_g_l']].round(4))
display(AFFINITY_CHEM_METRICS.round(4))


### Comparación de trayectorias hasta el final de cada LAB

Las dos figuras químicas separan glucosa, fructosa, G+F y etanol para identificar qué azúcar conserva la cola. Para LAB016–018 los puntos verdes provienen de densidad, no de análisis separados de G/F/E; su transformación lineal es una comparación orientativa. Para LAB019–021 se incluyen las muestras químicas disponibles. Las líneas verticales marcan los pulsos de N.

Las dos figuras gaseosas muestran, respectivamente, el ajuste global congelado y la transferencia con la capa gaseosa de `theta_full` entrenada en la otra corrida. Cada panel conserva el horizonte completo. Para juzgar una mejora deben revisarse conjuntamente el final del CO₂, la amplitud y el error químico: retrasar la curva no garantiza acercarla a los datos.


In [ ]:

AFFINITY_FIGURES_BEFORE=len(FIGURES)
for cohort,tag in [(COHORT2,'recent'),(COHORT1,'016_018')]:
    fig,axs=plt.subplots(3,4,figsize=(19,11),sharey='col')
    for i,lab in enumerate(cohort):
        for j,channel in enumerate(['G','F','GF','E']):
            ax=axs[i,j]
            for case,(label,color,ls) in AFFINITY_STYLES.items():
                cr=AFFINITY_CORES[case,lab]
                ax.plot(cr.index,cr[channel],label=label,color=color,ls=ls,lw=1.7)
            if lab in COHORT2:
                pts=CHEM[CHEM.lab.eq(lab)&~CHEM.is_NUT].dropna(subset=[channel+'_g_l'])
                ax.scatter(pts.t_h,pts[channel+'_g_l'],s=18,c='black',zorder=5,label='Medición')
            if channel=='GF':
                dp=DENSITY_POINTS[DENSITY_POINTS.lab.eq(lab)]
                ax.scatter(dp.t_h,dp.GF_density_g_l,s=20,c='#009E73',zorder=4,label='G+F por densidad')
            for event,dose in events_by_lab[lab]:ax.axvline(event,color='deeppink',ls=':',lw=.9)
            ax.set(title=lab+' · '+{'GF':'G+F','E':'Etanol'}.get(channel,channel),ylabel='g/L',
                xlim=(-3,BATCHES[lab].time[-1]+3))
            if i==2:ax.set_xlabel('t desde inoculación [h]')
            ax.grid(alpha=.3)
    legend_items={}
    for ax in axs[0]:
        handles,labels=ax.get_legend_handles_labels()
        for handle,label in zip(handles,labels):legend_items.setdefault(label,handle)
    fig.legend(list(legend_items.values()),list(legend_items),loc='upper center',ncol=4)
    fig.suptitle('Sustituciones de sF y sG · química · '+('LAB019–021' if tag=='recent' else 'LAB016–018'))
    show(fig,'sugar_affinity_parameter_comparison/chemistry_'+tag)
for mode in ('global','transfer'):
    fig,axs=plt.subplots(2,3,figsize=(17,9))
    for ax,lab in zip(axs.flat,ALL_LABS):
        d=OBS[OBS.lab.eq(lab)]
        ax.scatter(d.t_h,d.y,s=11,c='black',zorder=5,label='CO₂ observado')
        for case,(label,color,ls) in AFFINITY_STYLES.items():
            pr=AFFINITY_GAS[mode,case,lab]
            ax.plot(pr.t_h,pr.CO2,label=label,color=color,ls=ls,lw=1.7)
        for event,dose in events_by_lab[lab]:ax.axvline(event,color='deeppink',ls=':',lw=.9)
        ax.axhline(LOD,c='#bbbbbb',ls=':',lw=.7)
        title=lab
        if mode=='transfer':title+=' · entrenado en '+('019–021' if lab in COHORT1 else '016–018')
        ax.set(title=title,xlabel='t desde inoculación [h]',ylabel='g CO₂/L/h',
            xlim=(-3,BATCHES[lab].time[-1]+3))
        ax.grid(alpha=.3)
    handles,labels=axs[0,0].get_legend_handles_labels()
    fig.legend(handles,labels,loc='upper center',ncol=3)
    fig.suptitle('Sustituciones de sF y sG · '+('Z2 global fijo de theta_full' if mode=='global' else 'transferencia con Z2 fijo de theta_full'))
    show(fig,'sugar_affinity_parameter_comparison/co2_'+mode)
assert len(FIGURES)-AFFINITY_FIGURES_BEFORE==4


### Lectura cuantitativa y alcance de la prueba

La tabla siguiente compara cada sustitución con `theta_full` en el mismo LAB. Un incremento de G+F final indica consumo más lento. En las diferencias de RMSE, un valor negativo significa menor error. Si la caída no llega dentro del horizonte se deja sin valor, nunca se interpreta como cero.

Esta prueba distingue qué diferencias entre vectores explican el comportamiento, pero no estima valores óptimos. `theta_full` ya predice una caída demasiado tardía en algunos LAB, especialmente LAB021: ralentizar la captación puede empeorar allí el resultado. Además, la fuente gaseosa conserva su formulación actual, incluida la trayectoria sin nutrición. Una sustitución química por sí sola no valida esa formulación. Antes de adoptar cambios se deben contrastar G/F/E medidos y validar una eventual recalibración entre corridas.


In [ ]:

affinity_summary=AFFINITY_METRICS[AFFINITY_METRICS['mode'].eq('global')].copy()
baseline=affinity_summary[affinity_summary['case'].eq('full')].set_index('lab')
for col in ['GF_end_g_l','E_end_g_l','RMSE_CO2','RMSE_CO2_after_observed_peak','fall_h']:
    affinity_summary['change_'+col]=affinity_summary[col]-affinity_summary.lab.map(baseline[col])
affinity_chem=AFFINITY_CHEM_METRICS.pivot(index=['lab','case'],columns='channel',values='RMSE')
for channel in ['GF','E','N']:
    ref=AFFINITY_CHEM_METRICS[AFFINITY_CHEM_METRICS['case'].eq('full')&AFFINITY_CHEM_METRICS.channel.eq(channel)].set_index('lab').RMSE
    vals=affinity_chem[channel]-affinity_chem.index.get_level_values('lab').map(ref)
    changes=vals.rename('change_RMSE_'+channel).reset_index()
    affinity_summary=affinity_summary.merge(changes,on=['lab','case'],how='left',validate='one_to_one')
display(affinity_summary[['lab','case','change_GF_end_g_l','change_E_end_g_l',
    'change_RMSE_CO2','change_RMSE_CO2_after_observed_peak','change_fall_h',
    'change_RMSE_GF','change_RMSE_E','change_RMSE_N']].round(4))
for lab in ALL_LABS:
    d=affinity_summary[affinity_summary.lab.eq(lab)].set_index('case')
    recent=''
    if lab in COHORT2:
        recent=(f" Cambiar ambos modifica RMSE de G+F en {d.loc['full_sF_sG_default','change_RMSE_GF']:+.2f} g/L "
                f"y de etanol en {d.loc['full_sF_sG_default','change_RMSE_E']:+.2f} g/L.")
    display(Markdown(f"**{lab}.** G+F final con full: {d.loc['full','GF_end_g_l']:.2f} g/L; "
        f"con sF default: {d.loc['full_sF_default','GF_end_g_l']:.2f}; "
        f"con sG default: {d.loc['full_sG_default','GF_end_g_l']:.2f}; "
        f"con ambos: {d.loc['full_sF_sG_default','GF_end_g_l']:.2f}."+recent))
for name,frame in [('parameters',AFFINITY_PARAMETERS.rename_axis('parameter').reset_index()),
    ('gas_parameters_frozen',pd.DataFrame(AFFINITY_FIXED_GAS).rename_axis('parameter').reset_index()),
    ('metrics',AFFINITY_METRICS),('chemistry_metrics',AFFINITY_CHEM_METRICS),
    ('changes_from_full',affinity_summary),('checks',pd.DataFrame(AFFINITY_CHECKS)),
    ('gas_trajectories',pd.concat(AFFINITY_TRAJECTORIES,ignore_index=True)),
    ('chemistry_trajectories',pd.concat([cr.reset_index(names='t_h').assign(case=case,lab=lab)
        for (case,lab),cr in AFFINITY_CORES.items()],ignore_index=True))]:
    frame.to_csv(AFFINITY_OUT/(name+'.csv'),index=False)
affinity_manifest=dict(created_at=datetime.now().astimezone().isoformat(),
    diagnostic='sF and sG substitutions; frozen theta_full gas parameters',
    cases=AFFINITY_CASES,gas_parameters=AFFINITY_FIXED_GAS,
    chemical_parameters_refitted=False,gas_parameters_refitted=False,
    gas_freezing_policy='theta_full Z2 shared by all five cases; all_six for global, opposite cohort for transfer',
    chemistry_counterfactual_recomputed=True,activation='smoothstep predicted G+F consumption 2–10 g/L',
    horizons={lab:float(bt.time[-1]) for lab,bt in BATCHES.items()},
    observations_sha256=sha256(OUT/'observations.csv'),initial_conditions_sha256=sha256(OUT/'initial_conditions.csv'),
    existing_gas_parameters_sha256=sha256(OUT/'parameters.csv'),
    source_sha256=sha256(Path.cwd()/'co2_lab016_021_recalibration_v2.source.ipynb'),
    figures=FIGURES[AFFINITY_FIGURES_BEFORE:],
    checks=dict(max_balance_error=float(pd.DataFrame(AFFINITY_CHECKS).balance_error.max()),
        max_baseline_error=float(pd.DataFrame(AFFINITY_CHECKS).baseline_max_error.max())))
(AFFINITY_OUT/'manifest.json').write_text(json.dumps(affinity_manifest,indent=2,ensure_ascii=False),encoding='utf-8')
assert AFFINITY_FIXED_GAS==AFFINITY_GAS_SNAPSHOT and THETAS==THETA_SNAPSHOT and OLD==OLD_SNAPSHOT
print('Prueba controlada exportada; sin optimización adicional:',AFFINITY_OUT)


## 10. Exportación, trazabilidad y comprobaciones finales

Se guardan parámetros nuevos en una carpeta independiente, tablas, curvas, figuras e hipótesis. No se cambia el modelo compartido, los parámetros químicos ni los archivos crudos. La ganancia antigua no se usa como una calibración vigente: se muestra sólo como referencia. Estos nuevos parámetros son candidatos condicionados a esta arquitectura y a las entradas asumidas; su revisión debe considerar errores de forma, transferencia y límites activos antes de adoptarlos en el EKF.

In [ ]:
assert set(OBS.lab)==set(ALL_LABS) and not OBS.lab.isin([f'LAB{i:03}' for i in range(1,16)]).any()
assert THETAS==THETA_SNAPSHOT and OLD==OLD_SNAPSHOT
CHECK_TABLE=pd.DataFrame(CHECKS)
assert CHECK_TABLE.parity.max()<1e-12 and CHECK_TABLE.balance.max()<1e-10
assert all(sha256(ROOT/p)==h for p,h in HASHES_BEFORE.items()), 'Se modificó un archivo de entrada'
for name,df in [('parameters',FIT_TABLE),('multistart',pd.DataFrame(MULTISTART)),('metrics',METRICS),
    ('transfer_metrics',CV),('residuals',RESIDUALS),('chemistry_metrics',CHEM_METRICS),
    ('initial_conditions',INITIALS),('observations',OBS),('sensor_audit',SENSOR),
    ('chemistry_observations_corrected_YAN',CHEM),('checks',CHECK_TABLE),('numerical_refinement',REFINEMENT),('nitrogen_balance',N_BALANCE),('recent_observation_audit',RECENT_AUDIT),('recent_hourly_complete',RECENT_HOURLY),('recent_complete_coverage',RECENT_COVERAGE),('biomass_lab016_018',BIOMASS_016_018),('initial_sugar_from_density_lab016_018',INITIAL_SUGAR_016_018),('activation_times',ACTIVATION_TIMES),('activation_trajectories',ACTIVATION_TRAJECTORIES)]:
    df.to_csv(OUT/(name+'.csv'),index=False)
traj=[]
for (tn,lab,case),pr in PREDICTIONS.items():traj.append(pr.assign(theta=tn,lab=lab,case=case))
pd.concat(traj,ignore_index=True).to_csv(OUT/'gas_trajectories.csv',index=False)
pd.concat([cr.reset_index(names='t_h').assign(theta=tn,lab=lab) for (tn,lab),cr in CORES.items()],ignore_index=True).to_csv(OUT/'chemistry_trajectories.csv',index=False)
manifest=dict(created_at=datetime.now().astimezone().isoformat(),git_head=GIT_HEAD,remote=GIT_REMOTE,
    input_sha256=HASHES_BEFORE,fit_labs=list(ALL_LABS),excluded_CO2_labs='LAB001–015',
    theta_fixed=THETAS,old_gas_start=OLD,recent_gas_source=str(RECENT_CO2_PATH.relative_to(ROOT)),recent_temperature_source=str(RECENT_TEMP_PATH.relative_to(ROOT)),recent_horizon_policy='last chemical sample per LAB',recent_coverage=RECENT_COVERAGE.to_dict('records'),biomass_measurements_used_as_initials=False,initial_viable_lab016_018_million_cells_ml=INITIAL_VIABLE_016_018_MILLION_ML,initial_X_lab016_018_g_l=INITIAL_X_016_018_G_L,initial_biomass_source='user-specified viable concentration; no additional viability fraction',biomass_conversion_g_l_per_million_cells_ml=BIOMASS_G_L_PER_MILLION_ML,corrected_YAN_file=str(CORRECTED_YAN_PATH.relative_to(ROOT)),offset_in_model=False,initial_conditions=INITIALS.to_dict('records'),initial_GF_lab016_018_g_l=INITIAL_GF_016_018_G_L,initial_density_lab016_018_g_l=INITIAL_DENSITY_016_018_G_L,initial_GF_allocation='previous G:F ratio preserved',
    yield_CO2_E_fixed=joint.CO2_G_PER_G_ETHANOL,conversion_fixed=FACTOR,
    notebook_version='v2',baseline_source=str(V1_SOURCE_PATH.relative_to(ROOT)),baseline_source_sha256=V1_HASHES_BEFORE[str(V1_SOURCE_PATH.relative_to(ROOT))],baseline_files_sha256=V1_HASHES_BEFORE,
    gate='predicted S=G+F consumption smoothstep 2–10 g/L',sugar_activation_start_g_l=SUGAR_ACTIVATION_START_G_L,sugar_activation_full_g_l=SUGAR_ACTIVATION_FULL_G_L,ethanol_activation_condition=False,Cd0=0.,kernel='analytic_peak',
    loss='equal mean square weight per LAB; one-sided censor below 0.05 g/L/h',
    fit_stages={'Z1':['matrix_gain'],'Z2':list(BOUNDS)},bounds=BOUNDS,
    gas_integrator='implicit conservative Euler; exact positive quadratic root',density_sugar_comparison_path='density_sugar_comparison/manifest.json',sugar_affinity_diagnostic_path='sugar_affinity_parameter_comparison/manifest.json',figures=FIGURES,source_sha256=sha256(Path.cwd()/'co2_lab016_021_recalibration_v2.source.ipynb'))
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
assert np.array_equal(CHEM.YAN_mg_l.to_numpy(),CORRECTED_YAN.set_index(['lab','sample']).loc[
    pd.MultiIndex.from_frame(CHEM[['lab','sample']]),'YAN_mg_l'].to_numpy())
print('CSV corregido leído directamente; entradas originales intactas. Comprobaciones:')
display(CHECK_TABLE)
print('Resultados:',OUT)
assert all(sha256(ROOT/p)==h for p,h in V1_HASHES_BEFORE.items()), 'Se modificó el respaldo v1'
print('Respaldo v1 y sus resultados intactos. Salidas independientes:',OUT)
